In [2]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import SMOTE
import pickle

# ============================================================
# CONFIGURATION
# ============================================================
DATA_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "data")
OUTPUT_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

os.makedirs(OUTPUT_PATH, exist_ok=True)

print("=" * 60)
print("FIBROID PREDICTION - PREPROCESSING PIPELINE")
print("=" * 60)

# ============================================================
# STEP 1: LOAD CORE TABLES
# ============================================================
print("\n[1/8] Loading core tables...")

patients = pd.read_csv(os.path.join(DATA_PATH, "patients.csv"),
                       usecols=['subject_id', 'gender', 'anchor_age', 'anchor_year'])

admissions = pd.read_csv(os.path.join(DATA_PATH, "admissions.csv"),
                         usecols=['subject_id', 'hadm_id', 'admittime', 'race'])

diagnoses = pd.read_csv(os.path.join(DATA_PATH, "diagnoses_icd.csv"),
                        usecols=['subject_id', 'hadm_id', 'icd_code', 'icd_version'])

omr = pd.read_csv(os.path.join(DATA_PATH, "omr.csv"),
                  usecols=['subject_id', 'chartdate', 'result_name', 'result_value'])

print(f"  patients:    {len(patients):,} rows")
print(f"  admissions:  {len(admissions):,} rows")
print(f"  diagnoses:   {len(diagnoses):,} rows")
print(f"  omr:         {len(omr):,} rows")

# ============================================================
# STEP 2: FILTER FEMALE PATIENTS AGED 18-55
# ============================================================
print("\n[2/8] Filtering eligible patients (Female, age 18-55)...")

female_pts = patients[patients['gender'] == 'F'].copy()
female_pts = female_pts[(female_pts['anchor_age'] >= 18) &
                        (female_pts['anchor_age'] <= 55)]

print(f"  Eligible female patients (18-55): {len(female_pts):,}")

# ============================================================
# STEP 3: IDENTIFY FIBROID CASES
# ============================================================
print("\n[3/8] Identifying fibroid cases...")

fibroid_icd10 = diagnoses[
    (diagnoses['icd_version'] == 10) &
    (diagnoses['icd_code'].str.startswith('D25'))
]['subject_id'].unique()

fibroid_icd9 = diagnoses[
    (diagnoses['icd_version'] == 9) &
    (diagnoses['icd_code'].str.startswith('218'))
]['subject_id'].unique()

fibroid_subjects = set(fibroid_icd10) | set(fibroid_icd9)
print(f"  Fibroid cases found: {len(fibroid_subjects):,}")

# ============================================================
# STEP 4: BUILD COHORT WITH LABELS
# ============================================================
print("\n[4/8] Building labelled cohort...")

eligible_subjects = set(female_pts['subject_id'])
fibroid_eligible = fibroid_subjects & eligible_subjects

print(f"  Cases  (label=1): {len(fibroid_eligible):,}")
print(f"  Controls (label=0): {len(eligible_subjects - fibroid_eligible):,}")

admissions['admittime'] = pd.to_datetime(admissions['admittime'])
first_admissions = admissions.sort_values('admittime').groupby('subject_id').first().reset_index()
first_admissions = first_admissions[first_admissions['subject_id'].isin(eligible_subjects)]

first_admissions['label'] = first_admissions['subject_id'].apply(
    lambda x: 1 if x in fibroid_eligible else 0
)

first_admissions = first_admissions.merge(
    female_pts[['subject_id', 'anchor_age']], on='subject_id', how='left'
)
first_admissions.rename(columns={'anchor_age': 'age'}, inplace=True)
print(f"  Total cohort: {len(first_admissions):,}")

# ============================================================
# STEP 5: EXTRACT COMORBIDITIES
# ============================================================
print("\n[5/8] Extracting comorbidities...")

def get_comorbidity_flags(diagnoses_df, subject_ids):
    results = pd.DataFrame({'subject_id': list(subject_ids)})

    hyp10 = diagnoses_df[(diagnoses_df['icd_version']==10) &
                          (diagnoses_df['icd_code'].str.match(r'^I1[012345]'))]
    hyp9  = diagnoses_df[(diagnoses_df['icd_version']==9) &
                          (diagnoses_df['icd_code'].str.match(r'^40[12345]'))]
    results['hypertension'] = results['subject_id'].isin(
        set(hyp10['subject_id']) | set(hyp9['subject_id'])).astype(int)

    diab10 = diagnoses_df[(diagnoses_df['icd_version']==10) &
                           (diagnoses_df['icd_code'].str.match(r'^E1[01234]'))]
    diab9  = diagnoses_df[(diagnoses_df['icd_version']==9) &
                           (diagnoses_df['icd_code'].str.match(r'^250'))]
    results['diabetes'] = results['subject_id'].isin(
        set(diab10['subject_id']) | set(diab9['subject_id'])).astype(int)

    cad10 = diagnoses_df[(diagnoses_df['icd_version']==10) &
                          (diagnoses_df['icd_code'].str.match(r'^I2[12345]'))]
    cad9  = diagnoses_df[(diagnoses_df['icd_version']==9) &
                          (diagnoses_df['icd_code'].str.match(r'^41[01234]'))]
    results['coronary_artery_disease'] = results['subject_id'].isin(
        set(cad10['subject_id']) | set(cad9['subject_id'])).astype(int)

    return results

comorbidities = get_comorbidity_flags(diagnoses, set(first_admissions['subject_id']))
first_admissions = first_admissions.merge(comorbidities, on='subject_id', how='left')
print("  Done: hypertension, diabetes, coronary_artery_disease")

# ============================================================
# STEP 6: EXTRACT VITALS FROM OMR
# ============================================================
print("\n[6/8] Extracting vitals from OMR...")

omr_filtered = omr[omr['subject_id'].isin(set(first_admissions['subject_id']))].copy()
omr_filtered['result_value'] = pd.to_numeric(omr_filtered['result_value'], errors='coerce')

omr_mapping = {
    'BMI (kg/m2)': 'bmi',
    'Respiratory Rate': 'respiratory_rate',
    'Heart Rate': 'heart_rate',
    'O2 saturation pulseoxymetry': 'spo2',
    'Temperature Fahrenheit': 'temperature',
}

omr_filtered['feature'] = omr_filtered['result_name'].map(omr_mapping)
omr_filtered = omr_filtered[omr_filtered['feature'].notna()]

bp_rows = omr[omr['result_name'] == 'Blood Pressure'].copy()
bp_rows = bp_rows[bp_rows['subject_id'].isin(set(first_admissions['subject_id']))]

def extract_sbp(val):
    try: return float(str(val).split('/')[0])
    except: return np.nan

def extract_dbp(val):
    try: return float(str(val).split('/')[1])
    except: return np.nan

bp_rows['sbp'] = bp_rows['result_value'].apply(extract_sbp)
bp_rows['dbp'] = bp_rows['result_value'].apply(extract_dbp)

sbp_mean = bp_rows.groupby('subject_id')['sbp'].mean().reset_index()
sbp_mean.columns = ['subject_id', 'sbp']
dbp_mean = bp_rows.groupby('subject_id')['dbp'].mean().reset_index()
dbp_mean.columns = ['subject_id', 'dbp']

omr_pivot = omr_filtered.groupby(['subject_id', 'feature'])['result_value'].mean().unstack()
omr_pivot.reset_index(inplace=True)

first_admissions = first_admissions.merge(omr_pivot, on='subject_id', how='left')
first_admissions = first_admissions.merge(sbp_mean, on='subject_id', how='left')
first_admissions = first_admissions.merge(dbp_mean, on='subject_id', how='left')
print("  Done: BMI, SBP, DBP, HR, SpO2, Temperature, Respiratory Rate")

# ============================================================
# STEP 7: EXTRACT LAB VALUES FROM LABEVENTS (CHUNKED)
# ============================================================
print("\n[7/8] Extracting lab values from labevents (may take 10-20 mins)...")

LAB_ITEM_IDS = {
    51222: 'haemoglobin',
    51279: 'rbc',
    51301: 'wbc',
    51265: 'platelets',
    50931: 'glucose',
    50907: 'total_cholesterol',
    51000: 'triglycerides',
    50906: 'hdl_cholesterol',
    50862: 'albumin',
    51006: 'bun',
    50878: 'ast',
    50824: 'sodium',
    51237: 'inr',
}

target_subjects = set(first_admissions['subject_id'])
lab_results = {item_id: {} for item_id in LAB_ITEM_IDS}

CHUNK_SIZE = 500000
lab_file = os.path.join(DATA_PATH, "labevents.csv")

chunk_count = 0
for chunk in pd.read_csv(lab_file, chunksize=CHUNK_SIZE,
                         usecols=['subject_id', 'itemid', 'valuenum']):
    chunk_count += 1
    if chunk_count % 10 == 0:
        print(f"    Processing chunk {chunk_count}...")

    chunk = chunk[
        (chunk['subject_id'].isin(target_subjects)) &
        (chunk['itemid'].isin(LAB_ITEM_IDS.keys()))
    ]

    if len(chunk) == 0:
        continue

    for item_id in LAB_ITEM_IDS:
        rows = chunk[chunk['itemid'] == item_id][['subject_id', 'valuenum']].dropna()
        for _, row in rows.iterrows():
            sid = row['subject_id']
            val = row['valuenum']
            if sid not in lab_results[item_id]:
                lab_results[item_id][sid] = []
            lab_results[item_id][sid].append(val)

print(f"  Finished reading labevents ({chunk_count} chunks)")

lab_df = pd.DataFrame({'subject_id': list(target_subjects)})
for item_id, name in LAB_ITEM_IDS.items():
    subject_means = {sid: np.mean(vals) for sid, vals in lab_results[item_id].items()}
    lab_df[name] = lab_df['subject_id'].map(subject_means)

first_admissions = first_admissions.merge(lab_df, on='subject_id', how='left')
print("  Done: Hb, RBC, WBC, Platelets, Glucose, Cholesterol, Triglycerides, HDL, Albumin, BUN, AST, Sodium, INR")

# ============================================================
# STEP 8: ENCODE, CLEAN, SPLIT, SCALE, SMOTE
# ============================================================
print("\n[8/8] Encoding, cleaning, splitting, scaling, SMOTE...")

race_map = {
    'WHITE': 'white',
    'BLACK/AFRICAN AMERICAN': 'black',
    'HISPANIC/LATINO': 'hispanic',
    'ASIAN': 'asian',
}
first_admissions['race_clean'] = first_admissions['race'].str.upper().apply(
    lambda x: next((v for k, v in race_map.items() if k in str(x)), 'other')
)
race_dummies = pd.get_dummies(first_admissions['race_clean'], prefix='race', drop_first=True)
first_admissions = pd.concat([first_admissions, race_dummies], axis=1)

feature_cols = [
    'age', 'bmi', 'sbp', 'dbp', 'heart_rate', 'spo2',
    'temperature', 'respiratory_rate',
    'hypertension', 'diabetes', 'coronary_artery_disease',
    'haemoglobin', 'rbc', 'wbc', 'platelets', 'glucose',
    'total_cholesterol', 'triglycerides', 'hdl_cholesterol',
    'albumin', 'bun', 'ast', 'sodium', 'inr'
] + [c for c in race_dummies.columns]

feature_cols = [c for c in feature_cols if c in first_admissions.columns]
print(f"  Final features: {len(feature_cols)}")

df = first_admissions[feature_cols + ['label', 'subject_id']].copy()
threshold = len(feature_cols) * 0.5
df = df.dropna(thresh=int(threshold), subset=feature_cols)
print(f"  After dropping >50% missing: {len(df):,} patients")
print(f"  Cases: {df['label'].sum():,} | Controls: {(df['label']==0).sum():,}")

for col in feature_cols:
    df[col] = df[col].fillna(df[col].median())

race_col = first_admissions.set_index('subject_id')['race_clean']
df['race_clean'] = df['subject_id'].map(race_col)

X = df[feature_cols].values
y = df['label'].values
subject_ids = df['subject_id'].values
race_labels = df['race_clean'].values

X_train, X_temp, y_train, y_temp, sid_train, sid_temp, race_train, race_temp = train_test_split(
    X, y, subject_ids, race_labels,
    test_size=0.30, stratify=y, random_state=RANDOM_SEED
)
X_val, X_test, y_val, y_test, sid_val, sid_test, race_val, race_test = train_test_split(
    X_temp, y_temp, sid_temp, race_temp,
    test_size=0.50, stratify=y_temp, random_state=RANDOM_SEED
)

print(f"  Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print(f"  Before SMOTE — Cases: {y_train.sum():,} | Controls: {(y_train==0).sum():,}")
smote = SMOTE(random_state=RANDOM_SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)
print(f"  After SMOTE  — Cases: {y_train_sm.sum():,} | Controls: {(y_train_sm==0).sum():,}")

pd.DataFrame(X_train_sm, columns=feature_cols).to_csv(os.path.join(OUTPUT_PATH, "X_train.csv"), index=False)
pd.DataFrame(X_val_scaled, columns=feature_cols).to_csv(os.path.join(OUTPUT_PATH, "X_val.csv"), index=False)
pd.DataFrame(X_test_scaled, columns=feature_cols).to_csv(os.path.join(OUTPUT_PATH, "X_test.csv"), index=False)
pd.DataFrame({'label': y_train_sm}).to_csv(os.path.join(OUTPUT_PATH, "y_train.csv"), index=False)
pd.DataFrame({'label': y_val}).to_csv(os.path.join(OUTPUT_PATH, "y_val.csv"), index=False)
pd.DataFrame({'label': y_test}).to_csv(os.path.join(OUTPUT_PATH, "y_test.csv"), index=False)
pd.DataFrame({'subject_id': sid_test, 'race': race_test, 'label': y_test}).to_csv(
    os.path.join(OUTPUT_PATH, "test_demographics.csv"), index=False)

with open(os.path.join(OUTPUT_PATH, "scaler.pkl"), 'wb') as f:
    pickle.dump(scaler, f)
with open(os.path.join(OUTPUT_PATH, "feature_cols.pkl"), 'wb') as f:
    pickle.dump(feature_cols, f)

print("\n" + "=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"Output: {OUTPUT_PATH}")
print(f"Train: {X_train_sm.shape} | Val: {X_val_scaled.shape} | Test: {X_test_scaled.shape}")
print(f"Features: {len(feature_cols)} | Seed: {RANDOM_SEED}")
print("\nReady. Run 02_train_models.ipynb next.")

FIBROID PREDICTION - PREPROCESSING PIPELINE

[1/8] Loading core tables...
  patients:    364,627 rows
  admissions:  546,028 rows
  diagnoses:   6,364,488 rows
  omr:         7,753,027 rows

[2/8] Filtering eligible patients (Female, age 18-55)...
  Eligible female patients (18-55): 117,901

[3/8] Identifying fibroid cases...
  Fibroid cases found: 3,626

[4/8] Building labelled cohort...
  Cases  (label=1): 3,004
  Controls (label=0): 114,897
  Total cohort: 58,115

[5/8] Extracting comorbidities...
  Done: hypertension, diabetes, coronary_artery_disease

[6/8] Extracting vitals from OMR...
  Done: BMI, SBP, DBP, HR, SpO2, Temperature, Respiratory Rate

[7/8] Extracting lab values from labevents (may take 10-20 mins)...
    Processing chunk 10...
    Processing chunk 20...
    Processing chunk 30...
    Processing chunk 40...
    Processing chunk 50...
    Processing chunk 60...
    Processing chunk 70...
    Processing chunk 80...
    Processing chunk 90...
    Processing chunk 100..

KeyboardInterrupt: 

In [ ]:
# ============================================================
# MODEL 1: LOGISTIC REGRESSION (with Accuracy)
# ============================================================
import optuna
import pickle
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             f1_score, precision_score, brier_score_loss,
                             confusion_matrix, accuracy_score)
optuna.logging.set_verbosity(optuna.logging.WARNING)

OUTPUT_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")
RANDOM_SEED = 42

X_train = pd.read_csv(f"{OUTPUT_PATH}/X_train.csv").values
y_train = pd.read_csv(f"{OUTPUT_PATH}/y_train.csv").values.ravel()
X_val   = pd.read_csv(f"{OUTPUT_PATH}/X_val.csv").values
y_val   = pd.read_csv(f"{OUTPUT_PATH}/y_val.csv").values.ravel()
X_test  = pd.read_csv(f"{OUTPUT_PATH}/X_test.csv").values
y_test  = pd.read_csv(f"{OUTPUT_PATH}/y_test.csv").values.ravel()

print("=" * 60)
print("MODEL 1: LOGISTIC REGRESSION")
print("=" * 60)

def objective_lr(trial):
    C = trial.suggest_float('C', 1e-4, 100, log=True)
    solver = trial.suggest_categorical('solver', ['lbfgs', 'saga'])
    max_iter = trial.suggest_int('max_iter', 200, 1000)
    model = LogisticRegression(C=C, solver=solver, max_iter=max_iter,
                                random_state=RANDOM_SEED)
    model.fit(X_train, y_train)
    return roc_auc_score(y_val, model.predict_proba(X_val)[:,1])

study_lr = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_lr.optimize(objective_lr, n_trials=20)

print(f"  Best val AUC: {study_lr.best_value:.4f}")
print(f"  Best params: {study_lr.best_params}")

best = study_lr.best_params
lr_final = LogisticRegression(C=best['C'], solver=best['solver'],
                               max_iter=best['max_iter'],
                               random_state=RANDOM_SEED)
lr_final.fit(X_train, y_train)

y_prob = lr_final.predict_proba(X_test)[:,1]
y_pred = lr_final.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

results_lr = {
    'Model': 'Logistic Regression',
    'AUC_ROC': round(roc_auc_score(y_test, y_prob), 4),
    'AUC_PRC': round(average_precision_score(y_test, y_prob), 4),
    'Accuracy': round(accuracy_score(y_test, y_pred), 4),
    'F1': round(f1_score(y_test, y_pred), 4),
    'Sensitivity': round(tp/(tp+fn), 4),
    'Specificity': round(tn/(tn+fp), 4),
    'Precision': round(precision_score(y_test, y_pred), 4),
    'Brier_Score': round(brier_score_loss(y_test, y_prob), 4),
}

print("\n  TEST RESULTS:")
for k, v in results_lr.items():
    print(f"    {k}: {v}")

with open(f"{OUTPUT_PATH}/results_lr.pkl", 'wb') as f:
    pickle.dump(results_lr, f)

print("\nLogistic Regression DONE.")

In [ ]:
# ============================================================
# MODEL 2: XGBOOST
# ============================================================
import optuna
import pickle
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             f1_score, precision_score, brier_score_loss,
                             confusion_matrix, accuracy_score)
optuna.logging.set_verbosity(optuna.logging.WARNING)

OUTPUT_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")
RANDOM_SEED = 42

X_train = pd.read_csv(f"{OUTPUT_PATH}/X_train.csv").values
y_train = pd.read_csv(f"{OUTPUT_PATH}/y_train.csv").values.ravel()
X_val   = pd.read_csv(f"{OUTPUT_PATH}/X_val.csv").values
y_val   = pd.read_csv(f"{OUTPUT_PATH}/y_val.csv").values.ravel()
X_test  = pd.read_csv(f"{OUTPUT_PATH}/X_test.csv").values
y_test  = pd.read_csv(f"{OUTPUT_PATH}/y_test.csv").values.ravel()

print("=" * 60)
print("MODEL 2: XGBOOST")
print("=" * 60)

def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'random_state': RANDOM_SEED,
    }
    model = xgb.XGBClassifier(**params, eval_metric='logloss')
    model.fit(X_train, y_train)
    return roc_auc_score(y_val, model.predict_proba(X_val)[:,1])

study_xgb = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_xgb.optimize(objective_xgb, n_trials=20)

print(f"  Best val AUC: {study_xgb.best_value:.4f}")
print(f"  Best params: {study_xgb.best_params}")

best = study_xgb.best_params
xgb_final = xgb.XGBClassifier(**best, random_state=RANDOM_SEED, eval_metric='logloss')
xgb_final.fit(X_train, y_train)

y_prob = xgb_final.predict_proba(X_test)[:,1]
y_pred = xgb_final.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

results_xgb = {
    'Model': 'XGBoost',
    'AUC_ROC': round(roc_auc_score(y_test, y_prob), 4),
    'AUC_PRC': round(average_precision_score(y_test, y_prob), 4),
    'Accuracy': round(accuracy_score(y_test, y_pred), 4),
    'F1': round(f1_score(y_test, y_pred), 4),
    'Sensitivity': round(tp/(tp+fn), 4),
    'Specificity': round(tn/(tn+fp), 4),
    'Precision': round(precision_score(y_test, y_pred), 4),
    'Brier_Score': round(brier_score_loss(y_test, y_prob), 4),
}

print("\n  TEST RESULTS:")
for k, v in results_xgb.items():
    print(f"    {k}: {v}")

with open(f"{OUTPUT_PATH}/results_xgb.pkl", 'wb') as f:
    pickle.dump(results_xgb, f)

print("\nXGBoost DONE.")

In [ ]:
# ============================================================
# MODEL 3: MLP (Multilayer Perceptron)
# ============================================================
import optuna
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             f1_score, precision_score, brier_score_loss,
                             confusion_matrix, accuracy_score)
optuna.logging.set_verbosity(optuna.logging.WARNING)
torch.manual_seed(42)

OUTPUT_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")
RANDOM_SEED = 42

X_train = pd.read_csv(f"{OUTPUT_PATH}/X_train.csv").values.astype(np.float32)
y_train = pd.read_csv(f"{OUTPUT_PATH}/y_train.csv").values.ravel().astype(np.float32)
X_val   = pd.read_csv(f"{OUTPUT_PATH}/X_val.csv").values.astype(np.float32)
y_val   = pd.read_csv(f"{OUTPUT_PATH}/y_val.csv").values.ravel().astype(np.float32)
X_test  = pd.read_csv(f"{OUTPUT_PATH}/X_test.csv").values.astype(np.float32)
y_test  = pd.read_csv(f"{OUTPUT_PATH}/y_test.csv").values.ravel().astype(np.float32)

print("=" * 60)
print("MODEL 3: MLP")
print("=" * 60)

class MLPModel(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze()

def train_mlp(X_tr, y_tr, X_v, y_v, hidden1, hidden2, dropout, lr, epochs=30):
    model = MLPModel(X_tr.shape[1], hidden1, hidden2, dropout)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    
    train_ds = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
    train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)
    
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
    
    model.eval()
    with torch.no_grad():
        val_prob = model(torch.tensor(X_v)).numpy()
    return model, val_prob

def objective_mlp(trial):
    hidden1  = trial.suggest_categorical('hidden1', [64, 128, 256])
    hidden2  = trial.suggest_categorical('hidden2', [32, 64, 128])
    dropout  = trial.suggest_float('dropout', 0.1, 0.5)
    lr       = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    _, val_prob = train_mlp(X_train, y_train, X_val, y_val,
                             hidden1, hidden2, dropout, lr)
    return roc_auc_score(y_val, val_prob)

study_mlp = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_mlp.optimize(objective_mlp, n_trials=20)

print(f"  Best val AUC: {study_mlp.best_value:.4f}")
print(f"  Best params: {study_mlp.best_params}")

best = study_mlp.best_params
mlp_final, _ = train_mlp(X_train, y_train, X_val, y_val,
                          best['hidden1'], best['hidden2'],
                          best['dropout'], best['lr'], epochs=50)

mlp_final.eval()
with torch.no_grad():
    y_prob = mlp_final(torch.tensor(X_test)).numpy()

y_pred = (y_prob >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

results_mlp = {
    'Model': 'MLP',
    'AUC_ROC': round(roc_auc_score(y_test, y_prob), 4),
    'AUC_PRC': round(average_precision_score(y_test, y_prob), 4),
    'Accuracy': round(accuracy_score(y_test, y_pred), 4),
    'F1': round(f1_score(y_test, y_pred), 4),
    'Sensitivity': round(tp/(tp+fn), 4),
    'Specificity': round(tn/(tn+fp), 4),
    'Precision': round(precision_score(y_test, y_pred), 4),
    'Brier_Score': round(brier_score_loss(y_test, y_prob), 4),
}

print("\n  TEST RESULTS:")
for k, v in results_mlp.items():
    print(f"    {k}: {v}")

with open(f"{OUTPUT_PATH}/results_mlp.pkl", 'wb') as f:
    pickle.dump(results_mlp, f)

print("\nMLP DONE.")

In [ ]:
# ============================================================
# MODEL 4: TabFCN (Tabular Fully Connected Network)
# ============================================================
import optuna
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             f1_score, precision_score, brier_score_loss,
                             confusion_matrix, accuracy_score)
optuna.logging.set_verbosity(optuna.logging.WARNING)
torch.manual_seed(42)

OUTPUT_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")
RANDOM_SEED = 42

X_train = pd.read_csv(f"{OUTPUT_PATH}/X_train.csv").values.astype(np.float32)
y_train = pd.read_csv(f"{OUTPUT_PATH}/y_train.csv").values.ravel().astype(np.float32)
X_val   = pd.read_csv(f"{OUTPUT_PATH}/X_val.csv").values.astype(np.float32)
y_val   = pd.read_csv(f"{OUTPUT_PATH}/y_val.csv").values.ravel().astype(np.float32)
X_test  = pd.read_csv(f"{OUTPUT_PATH}/X_test.csv").values.astype(np.float32)
y_test  = pd.read_csv(f"{OUTPUT_PATH}/y_test.csv").values.ravel().astype(np.float32)

print("=" * 60)
print("MODEL 4: TabFCN")
print("=" * 60)

class TabFCN(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h),
                       nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers += [nn.Linear(in_dim, 1), nn.Sigmoid()]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze()

def train_tabfcn(X_tr, y_tr, X_v, y_v, hidden_dims, dropout, lr, epochs=40):
    model = TabFCN(X_tr.shape[1], hidden_dims, dropout)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCELoss()
    train_ds = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
    train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        val_prob = model(torch.tensor(X_v)).numpy()
    return model, val_prob

def objective_tabfcn(trial):
    n_layers = trial.suggest_int('n_layers', 2, 4)
    hidden_dim = trial.suggest_categorical('hidden_dim', [64, 128, 256, 512])
    hidden_dims = [hidden_dim] * n_layers
    dropout = trial.suggest_float('dropout', 0.1, 0.5)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    _, val_prob = train_tabfcn(X_train, y_train, X_val, y_val,
                                hidden_dims, dropout, lr)
    return roc_auc_score(y_val, val_prob)

study_tabfcn = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_tabfcn.optimize(objective_tabfcn, n_trials=20)

print(f"  Best val AUC: {study_tabfcn.best_value:.4f}")
print(f"  Best params: {study_tabfcn.best_params}")

best = study_tabfcn.best_params
hidden_dims = [best['hidden_dim']] * best['n_layers']
tabfcn_final, _ = train_tabfcn(X_train, y_train, X_val, y_val,
                                hidden_dims, best['dropout'], best['lr'], epochs=60)

tabfcn_final.eval()
with torch.no_grad():
    y_prob = tabfcn_final(torch.tensor(X_test)).numpy()

y_pred = (y_prob >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

results_tabfcn = {
    'Model': 'TabFCN',
    'AUC_ROC': round(roc_auc_score(y_test, y_prob), 4),
    'AUC_PRC': round(average_precision_score(y_test, y_prob), 4),
    'Accuracy': round(accuracy_score(y_test, y_pred), 4),
    'F1': round(f1_score(y_test, y_pred), 4),
    'Sensitivity': round(tp/(tp+fn), 4),
    'Specificity': round(tn/(tn+fp), 4),
    'Precision': round(precision_score(y_test, y_pred), 4),
    'Brier_Score': round(brier_score_loss(y_test, y_prob), 4),
}

print("\n  TEST RESULTS:")
for k, v in results_tabfcn.items():
    print(f"    {k}: {v}")

with open(f"{OUTPUT_PATH}/results_tabfcn.pkl", 'wb') as f:
    pickle.dump(results_tabfcn, f)

print("\nTabFCN DONE.")

In [ ]:
# ============================================================
# MODELS 5-8: SVM, TabNet, TCN, LSTM
# ============================================================
import optuna
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.svm import SVC
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             f1_score, precision_score, brier_score_loss,
                             confusion_matrix, accuracy_score)
from pytorch_tabnet.tab_model import TabNetClassifier
optuna.logging.set_verbosity(optuna.logging.WARNING)
torch.manual_seed(42)
np.random.seed(42)

OUTPUT_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")
RANDOM_SEED = 42

X_train = pd.read_csv(f"{OUTPUT_PATH}/X_train.csv").values.astype(np.float32)
y_train = pd.read_csv(f"{OUTPUT_PATH}/y_train.csv").values.ravel()
X_val   = pd.read_csv(f"{OUTPUT_PATH}/X_val.csv").values.astype(np.float32)
y_val   = pd.read_csv(f"{OUTPUT_PATH}/y_val.csv").values.ravel()
X_test  = pd.read_csv(f"{OUTPUT_PATH}/X_test.csv").values.astype(np.float32)
y_test  = pd.read_csv(f"{OUTPUT_PATH}/y_test.csv").values.ravel()

all_results = []

# ============================================================
# MODEL 5: SVM
# ============================================================
print("=" * 60)
print("MODEL 5: SVM")
print("=" * 60)

def objective_svm(trial):
    C = trial.suggest_float('C', 0.01, 100, log=True)
    kernel = trial.suggest_categorical('kernel', ['rbf', 'linear'])
    gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])
    model = SVC(C=C, kernel=kernel, gamma=gamma,
                probability=True, random_state=RANDOM_SEED)
    # Use subset for speed during tuning
    idx = np.random.choice(len(X_train), 10000, replace=False)
    model.fit(X_train[idx], y_train[idx])
    return roc_auc_score(y_val, model.predict_proba(X_val)[:,1])

study_svm = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_svm.optimize(objective_svm, n_trials=20)

print(f"  Best val AUC: {study_svm.best_value:.4f}")
print(f"  Best params: {study_svm.best_params}")

best = study_svm.best_params
svm_final = SVC(C=best['C'], kernel=best['kernel'], gamma=best['gamma'],
                probability=True, random_state=RANDOM_SEED)
svm_final.fit(X_train, y_train)

y_prob = svm_final.predict_proba(X_test)[:,1]
y_pred = svm_final.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

results_svm = {
    'Model': 'SVM',
    'AUC_ROC': round(roc_auc_score(y_test, y_prob), 4),
    'AUC_PRC': round(average_precision_score(y_test, y_prob), 4),
    'Accuracy': round(accuracy_score(y_test, y_pred), 4),
    'F1': round(f1_score(y_test, y_pred), 4),
    'Sensitivity': round(tp/(tp+fn), 4),
    'Specificity': round(tn/(tn+fp), 4),
    'Precision': round(precision_score(y_test, y_pred), 4),
    'Brier_Score': round(brier_score_loss(y_test, y_prob), 4),
}
print("\n  TEST RESULTS:")
for k, v in results_svm.items():
    print(f"    {k}: {v}")
with open(f"{OUTPUT_PATH}/results_svm.pkl", 'wb') as f:
    pickle.dump(results_svm, f)
all_results.append(results_svm)
print("\nSVM DONE\n")

# ============================================================
# MODEL 6: TabNet
# ============================================================
print("=" * 60)
print("MODEL 6: TabNet")
print("=" * 60)

def objective_tabnet(trial):
    n_d = trial.suggest_int('n_d', 8, 64)
    n_steps = trial.suggest_int('n_steps', 3, 7)
    gamma = trial.suggest_float('gamma', 1.0, 2.0)
    lr = trial.suggest_float('lr', 1e-3, 1e-1, log=True)

    model = TabNetClassifier(
        n_d=n_d, n_a=n_d, n_steps=n_steps, gamma=gamma,
        optimizer_params=dict(lr=lr),
        verbose=0, seed=RANDOM_SEED
    )
    model.fit(
        X_train, y_train.astype(int),
        eval_set=[(X_val, y_val.astype(int))],
        max_epochs=50, patience=10, batch_size=1024
    )
    prob = model.predict_proba(X_val)[:,1]
    return roc_auc_score(y_val, prob)

study_tabnet = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_tabnet.optimize(objective_tabnet, n_trials=20)

print(f"  Best val AUC: {study_tabnet.best_value:.4f}")
print(f"  Best params: {study_tabnet.best_params}")

best = study_tabnet.best_params
tabnet_final = TabNetClassifier(
    n_d=best['n_d'], n_a=best['n_d'],
    n_steps=best['n_steps'], gamma=best['gamma'],
    optimizer_params=dict(lr=best['lr']),
    verbose=0, seed=RANDOM_SEED
)
tabnet_final.fit(
    X_train, y_train.astype(int),
    eval_set=[(X_val, y_val.astype(int))],
    max_epochs=100, patience=15, batch_size=1024
)

y_prob = tabnet_final.predict_proba(X_test)[:,1]
y_pred = tabnet_final.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

results_tabnet = {
    'Model': 'TabNet',
    'AUC_ROC': round(roc_auc_score(y_test, y_prob), 4),
    'AUC_PRC': round(average_precision_score(y_test, y_prob), 4),
    'Accuracy': round(accuracy_score(y_test, y_pred), 4),
    'F1': round(f1_score(y_test, y_pred), 4),
    'Sensitivity': round(tp/(tp+fn), 4),
    'Specificity': round(tn/(tn+fp), 4),
    'Precision': round(precision_score(y_test, y_pred), 4),
    'Brier_Score': round(brier_score_loss(y_test, y_prob), 4),
}
print("\n  TEST RESULTS:")
for k, v in results_tabnet.items():
    print(f"    {k}: {v}")
with open(f"{OUTPUT_PATH}/results_tabnet.pkl", 'wb') as f:
    pickle.dump(results_tabnet, f)
all_results.append(results_tabnet)
print("\nTabNet DONE\n")

# ============================================================
# MODEL 7: TCN (Temporal Convolutional Network)
# ============================================================
print("=" * 60)
print("MODEL 7: TCN")
print("=" * 60)

class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout):
        super().__init__()
        self.conv1 = nn.utils.weight_norm(nn.Conv1d(
            n_inputs, n_outputs, kernel_size, stride=stride,
            padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.utils.weight_norm(nn.Conv1d(
            n_outputs, n_outputs, kernel_size, stride=stride,
            padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                  self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCNModel(nn.Module):
    def __init__(self, input_size, num_channels, kernel_size, dropout):
        super().__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation = 2 ** i
            in_ch = input_size if i == 0 else num_channels[i-1]
            out_ch = num_channels[i]
            padding = (kernel_size - 1) * dilation
            layers.append(TemporalBlock(in_ch, out_ch, kernel_size, 1,
                                         dilation, padding, dropout))
        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.tcn(x)
        out = self.fc(out[:, :, -1])
        return self.sigmoid(out).squeeze()

def train_tcn(X_tr, y_tr, X_v, y_v, num_channels, kernel_size, dropout, lr, epochs=30):
    X_tr_t = torch.tensor(X_tr).unsqueeze(2)
    X_v_t  = torch.tensor(X_v).unsqueeze(2)
    model = TCNModel(1, num_channels, kernel_size, dropout)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    ds = TensorDataset(X_tr_t, torch.tensor(y_tr))
    dl = DataLoader(ds, batch_size=256, shuffle=True)

    for epoch in range(epochs):
        model.train()
        for xb, yb in dl:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        val_prob = model(X_v_t).numpy()
    return model, val_prob

def objective_tcn(trial):
    n_channels = trial.suggest_int('n_channels', 16, 64)
    n_levels   = trial.suggest_int('n_levels', 2, 4)
    kernel_size = trial.suggest_categorical('kernel_size', [3, 5, 7])
    dropout    = trial.suggest_float('dropout', 0.1, 0.4)
    lr         = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    num_channels = [n_channels] * n_levels
    _, val_prob = train_tcn(X_train, y_train, X_val, y_val,
                             num_channels, kernel_size, dropout, lr)
    return roc_auc_score(y_val, val_prob)

study_tcn = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_tcn.optimize(objective_tcn, n_trials=20)

print(f"  Best val AUC: {study_tcn.best_value:.4f}")
print(f"  Best params: {study_tcn.best_params}")

best = study_tcn.best_params
num_channels = [best['n_channels']] * best['n_levels']
tcn_final, _ = train_tcn(X_train, y_train, X_val, y_val,
                          num_channels, best['kernel_size'],
                          best['dropout'], best['lr'], epochs=50)

tcn_final.eval()
X_test_t = torch.tensor(X_test).unsqueeze(2)
with torch.no_grad():
    y_prob = tcn_final(X_test_t).numpy()

y_pred = (y_prob >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

results_tcn = {
    'Model': 'TCN',
    'AUC_ROC': round(roc_auc_score(y_test, y_prob), 4),
    'AUC_PRC': round(average_precision_score(y_test, y_prob), 4),
    'Accuracy': round(accuracy_score(y_test, y_pred), 4),
    'F1': round(f1_score(y_test, y_pred), 4),
    'Sensitivity': round(tp/(tp+fn), 4),
    'Specificity': round(tn/(tn+fp), 4),
    'Precision': round(precision_score(y_test, y_pred), 4),
    'Brier_Score': round(brier_score_loss(y_test, y_prob), 4),
}
print("\n  TEST RESULTS:")
for k, v in results_tcn.items():
    print(f"    {k}: {v}")
with open(f"{OUTPUT_PATH}/results_tcn.pkl", 'wb') as f:
    pickle.dump(results_tcn, f)
all_results.append(results_tcn)
print("\nTCN DONE\n")

# ============================================================
# MODEL 8: LSTM
# ============================================================
print("=" * 60)
print("MODEL 8: LSTM")
print("=" * 60)

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return self.sigmoid(out).squeeze()

def train_lstm(X_tr, y_tr, X_v, y_v, hidden_size, num_layers, dropout, lr, epochs=30):
    X_tr_t = torch.tensor(X_tr).unsqueeze(1)
    X_v_t  = torch.tensor(X_v).unsqueeze(1)
    model = LSTMModel(X_tr.shape[1], hidden_size, num_layers, dropout)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    ds = TensorDataset(X_tr_t, torch.tensor(y_tr))
    dl = DataLoader(ds, batch_size=256, shuffle=True)

    for epoch in range(epochs):
        model.train()
        for xb, yb in dl:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        val_prob = model(X_v_t).numpy()
    return model, val_prob

def objective_lstm(trial):
    hidden_size = trial.suggest_categorical('hidden_size', [32, 64, 128])
    num_layers  = trial.suggest_int('num_layers', 1, 3)
    dropout     = trial.suggest_float('dropout', 0.1, 0.5)
    lr          = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    _, val_prob = train_lstm(X_train, y_train, X_val, y_val,
                              hidden_size, num_layers, dropout, lr)
    return roc_auc_score(y_val, val_prob)

study_lstm = optuna.create_study(direction='maximize',
                                  sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_lstm.optimize(objective_lstm, n_trials=20)

print(f"  Best val AUC: {study_lstm.best_value:.4f}")
print(f"  Best params: {study_lstm.best_params}")

best = study_lstm.best_params
lstm_final, _ = train_lstm(X_train, y_train, X_val, y_val,
                            best['hidden_size'], best['num_layers'],
                            best['dropout'], best['lr'], epochs=50)

lstm_final.eval()
X_test_t = torch.tensor(X_test).unsqueeze(1)
with torch.no_grad():
    y_prob = lstm_final(X_test_t).numpy()

y_pred = (y_prob >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

results_lstm = {
    'Model': 'LSTM',
    'AUC_ROC': round(roc_auc_score(y_test, y_prob), 4),
    'AUC_PRC': round(average_precision_score(y_test, y_prob), 4),
    'Accuracy': round(accuracy_score(y_test, y_pred), 4),
    'F1': round(f1_score(y_test, y_pred), 4),
    'Sensitivity': round(tp/(tp+fn), 4),
    'Specificity': round(tn/(tn+fp), 4),
    'Precision': round(precision_score(y_test, y_pred), 4),
    'Brier_Score': round(brier_score_loss(y_test, y_prob), 4),
}
print("\n  TEST RESULTS:")
for k, v in results_lstm.items():
    print(f"    {k}: {v}")
with open(f"{OUTPUT_PATH}/results_lstm.pkl", 'wb') as f:
    pickle.dump(results_lstm, f)
all_results.append(results_lstm)
print("\nLSTM DONE\n")

# ============================================================
# FINAL SUMMARY TABLE
# ============================================================
print("=" * 60)
print("ALL MODELS COMPLETE - SUMMARY TABLE")
print("=" * 60)

summary = pd.DataFrame(all_results)
print(summary.to_string(index=False))
summary.to_csv(f"{OUTPUT_PATH}/all_results_summary.csv", index=False)
print(f"\nResults saved to {OUTPUT_PATH}/all_results_summary.csv")
print("\nAll 8 models done. Come back in the morning!")

In [ ]:
# ============================================================
# CELL: TCN + LSTM (Fixed)
# ============================================================
import numpy as np
import pandas as pd
import pickle, os, warnings
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import optuna
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             accuracy_score, f1_score, recall_score,
                             precision_score, brier_score_loss)
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

PROCESSED = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")

# Load data
X_train = pd.read_csv(os.path.join(PROCESSED, "X_train.csv")).values.astype(np.float32)
X_val   = pd.read_csv(os.path.join(PROCESSED, "X_val.csv")).values.astype(np.float32)
X_test  = pd.read_csv(os.path.join(PROCESSED, "X_test.csv")).values.astype(np.float32)
y_train = pd.read_csv(os.path.join(PROCESSED, "y_train.csv")).values.ravel().astype(np.float32)
y_val   = pd.read_csv(os.path.join(PROCESSED, "y_val.csv")).values.ravel().astype(np.float32)
y_test  = pd.read_csv(os.path.join(PROCESSED, "y_test.csv")).values.ravel().astype(np.float32)

input_dim = X_train.shape[1]  # 24

def make_loader(X, y, batch_size=256, shuffle=False):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

def evaluate(model, loader):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb).squeeze()
            probs.extend(out.numpy())
            labels.extend(yb.numpy())
    probs, labels = np.array(probs), np.array(labels)
    preds = (probs >= 0.5).astype(int)
    return {
        'AUC_ROC':     roc_auc_score(labels, probs),
        'AUC_PRC':     average_precision_score(labels, probs),
        'Accuracy':    accuracy_score(labels, preds),
        'F1':          f1_score(labels, preds, zero_division=0),
        'Sensitivity': recall_score(labels, preds, zero_division=0),
        'Specificity': recall_score(labels, preds, pos_label=0, zero_division=0),
        'Precision':   precision_score(labels, preds, zero_division=0),
        'Brier_Score': brier_score_loss(labels, probs),
    }

# ─────────────────────────────────────────────
# TCN
# ─────────────────────────────────────────────
print("=" * 60)
print("MODEL 7: TCN")
print("=" * 60)

class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.utils.weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation)),
            Chomp1d(padding),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.utils.weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation)),
            Chomp1d(padding),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCNModel(nn.Module):
    # Input shape: (batch, features, 1) — features as channels, length=1
    def __init__(self, input_dim, n_channels, n_levels, kernel_size, dropout):
        super().__init__()
        channels = [n_channels] * n_levels
        layers = []
        for i, out_ch in enumerate(channels):
            in_ch = input_dim if i == 0 else channels[i-1]
            dilation = 2 ** i
            padding = (kernel_size - 1) * dilation
            layers.append(TemporalBlock(in_ch, out_ch, kernel_size, 1, dilation, padding, dropout))
        self.net = nn.Sequential(*layers)
        self.fc = nn.Linear(n_channels, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        # x: (batch, features, 1)
        out = self.net(x)          # (batch, n_channels, 1)
        out = out[:, :, -1]        # (batch, n_channels)
        return self.sigmoid(self.fc(out))

def tcn_objective(trial):
    n_channels  = trial.suggest_categorical('n_channels', [16, 32, 64])
    n_levels    = trial.suggest_int('n_levels', 2, 4)
    kernel_size = trial.suggest_categorical('kernel_size', [2, 3])
    dropout     = trial.suggest_float('dropout', 0.1, 0.5)
    lr          = trial.suggest_float('lr', 1e-4, 1e-2, log=True)

    # Shape: (batch, input_dim, 1) — input_dim as channels
    train_loader = make_loader(X_train.reshape(-1, input_dim, 1), y_train, shuffle=True)
    val_loader   = make_loader(X_val.reshape(-1, input_dim, 1),   y_val)

    model = TCNModel(input_dim, n_channels, n_levels, kernel_size, dropout)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    crit  = nn.BCELoss()

    best_auc = 0
    for epoch in range(30):
        model.train()
        for xb, yb in train_loader:
            opt.zero_grad()
            loss = crit(model(xb).squeeze(), yb)
            loss.backward()
            opt.step()
        metrics = evaluate(model, val_loader)
        if metrics['AUC_ROC'] > best_auc:
            best_auc = metrics['AUC_ROC']
    return best_auc

study_tcn = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=42))
study_tcn.optimize(tcn_objective, n_trials=20, show_progress_bar=True)
print("Best val AUC:", study_tcn.best_value)
print("Best params:", study_tcn.best_params)

# Retrain best TCN
bp = study_tcn.best_params
train_loader = make_loader(X_train.reshape(-1, input_dim, 1), y_train, shuffle=True)
val_loader   = make_loader(X_val.reshape(-1, input_dim, 1),   y_val)
test_loader  = make_loader(X_test.reshape(-1, input_dim, 1),  y_test)

best_tcn = TCNModel(input_dim, bp['n_channels'], bp['n_levels'], bp['kernel_size'], bp['dropout'])
opt  = torch.optim.Adam(best_tcn.parameters(), lr=bp['lr'])
crit = nn.BCELoss()
best_auc, patience = 0, 10
for epoch in range(100):
    best_tcn.train()
    for xb, yb in train_loader:
        opt.zero_grad()
        loss = crit(best_tcn(xb).squeeze(), yb)
        loss.backward()
        opt.step()
    auc = evaluate(best_tcn, val_loader)['AUC_ROC']
    if auc > best_auc:
        best_auc = auc
        torch.save(best_tcn.state_dict(), os.path.join(PROCESSED, "tcn_best.pt"))
        patience = 10
    else:
        patience -= 1
        if patience == 0:
            break

best_tcn.load_state_dict(torch.load(os.path.join(PROCESSED, "tcn_best.pt"), weights_only=False))
tcn_results = evaluate(best_tcn, test_loader)
tcn_results['Model'] = 'TCN'
with open(os.path.join(PROCESSED, "tcn_results.pkl"), "wb") as f:
    pickle.dump(tcn_results, f)
print("\nTEST RESULTS:")
for k, v in tcn_results.items():
    if k != 'Model':
        print(f"  {k}: {v:.4f}")
print("\nTCN DONE")

# ─────────────────────────────────────────────
# LSTM
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("MODEL 8: LSTM")
print("=" * 60)

class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        # x: (batch, 1, input_dim)
        out, _ = self.lstm(x)
        return self.sigmoid(self.fc(out[:, -1, :]))

def lstm_objective(trial):
    hidden_size = trial.suggest_categorical('hidden_size', [32, 64, 128])
    num_layers  = trial.suggest_int('num_layers', 1, 3)
    dropout     = trial.suggest_float('dropout', 0.1, 0.5)
    lr          = trial.suggest_float('lr', 1e-4, 1e-2, log=True)

    # Shape: (batch, 1, input_dim)
    train_loader = make_loader(X_train.reshape(-1, 1, input_dim), y_train, shuffle=True)
    val_loader   = make_loader(X_val.reshape(-1, 1, input_dim),   y_val)

    model = LSTMModel(input_dim, hidden_size, num_layers, dropout)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    crit  = nn.BCELoss()

    best_auc = 0
    for epoch in range(30):
        model.train()
        for xb, yb in train_loader:
            opt.zero_grad()
            loss = crit(model(xb).squeeze(), yb)
            loss.backward()
            opt.step()
        metrics = evaluate(model, val_loader)
        if metrics['AUC_ROC'] > best_auc:
            best_auc = metrics['AUC_ROC']
    return best_auc

study_lstm = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
study_lstm.optimize(lstm_objective, n_trials=20, show_progress_bar=True)
print("Best val AUC:", study_lstm.best_value)
print("Best params:", study_lstm.best_params)

# Retrain best LSTM
bp = study_lstm.best_params
train_loader = make_loader(X_train.reshape(-1, 1, input_dim), y_train, shuffle=True)
val_loader   = make_loader(X_val.reshape(-1, 1, input_dim),   y_val)
test_loader  = make_loader(X_test.reshape(-1, 1, input_dim),  y_test)

best_lstm = LSTMModel(input_dim, bp['hidden_size'], bp['num_layers'], bp['dropout'])
opt  = torch.optim.Adam(best_lstm.parameters(), lr=bp['lr'])
crit = nn.BCELoss()
best_auc, patience = 0, 10
for epoch in range(100):
    best_lstm.train()
    for xb, yb in train_loader:
        opt.zero_grad()
        loss = crit(best_lstm(xb).squeeze(), yb)
        loss.backward()
        opt.step()
    auc = evaluate(best_lstm, val_loader)['AUC_ROC']
    if auc > best_auc:
        best_auc = auc
        torch.save(best_lstm.state_dict(), os.path.join(PROCESSED, "lstm_best.pt"))
        patience = 10
    else:
        patience -= 1
        if patience == 0:
            break

best_lstm.load_state_dict(torch.load(os.path.join(PROCESSED, "lstm_best.pt"), weights_only=False))
lstm_results = evaluate(best_lstm, test_loader)
lstm_results['Model'] = 'LSTM'
with open(os.path.join(PROCESSED, "lstm_results.pkl"), "wb") as f:
    pickle.dump(lstm_results, f)
print("\nTEST RESULTS:")
for k, v in lstm_results.items():
    if k != 'Model':
        print(f"  {k}: {v:.4f}")
print("\nLSTM DONE")

# ─────────────────────────────────────────────
# FINAL SUMMARY TABLE
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("FINAL SUMMARY — ALL 8 MODELS")
print("=" * 60)

result_files = {
    'Logistic Regression': 'lr_results.pkl',
    'XGBoost':             'xgb_results.pkl',
    'MLP':                 'mlp_results.pkl',
    'TabFCN':              'tabfcn_results.pkl',
    'SVM':                 'svm_results.pkl',
    'TabNet':              'tabnet_results.pkl',
    'TCN':                 'tcn_results.pkl',
    'LSTM':                'lstm_results.pkl',
}

rows = []
for name, fname in result_files.items():
    path = os.path.join(PROCESSED, fname)
    if os.path.exists(path):
        with open(path, 'rb') as f:
            r = pickle.load(f)
        rows.append({
            'Model':       name,
            'AUC-ROC':     round(r.get('AUC_ROC', float('nan')), 4),
            'AUC-PRC':     round(r.get('AUC_PRC', float('nan')), 4),
            'Accuracy':    round(r.get('Accuracy', float('nan')), 4),
            'F1':          round(r.get('F1', float('nan')), 4),
            'Sensitivity': round(r.get('Sensitivity', float('nan')), 4),
            'Specificity': round(r.get('Specificity', float('nan')), 4),
            'Precision':   round(r.get('Precision', float('nan')), 4),
            'Brier Score': round(r.get('Brier_Score', float('nan')), 4),
        })
    else:
        print(f"  WARNING: {fname} not found — skipping")

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))
df_results.to_csv(os.path.join(PROCESSED, "all_results_summary.csv"), index=False)
print("\nSaved to all_results_summary.csv")

In [ ]:
# ============================================================
# CELL: SHAP Interpretability (XGBoost - Best Model)
# ============================================================
import numpy as np
import pandas as pd
import pickle, os, warnings
import shap
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

PROCESSED  = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")
OUTPUT_FIG = os.path.join(os.getcwd(), "figures")

X_train = pd.read_csv(os.path.join(PROCESSED, "X_train.csv"))
X_test  = pd.read_csv(os.path.join(PROCESSED, "X_test.csv"))
y_train = pd.read_csv(os.path.join(PROCESSED, "y_train.csv")).values.ravel()
y_test  = pd.read_csv(os.path.join(PROCESSED, "y_test.csv")).values.ravel()

feature_names = list(X_train.columns)

# Retrain XGBoost with best params
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=1,
    eval_metric='logloss',
    random_state=42, n_jobs=-1
)
xgb_model.fit(X_train.values, y_train)
print("XGBoost retrained.")

# SHAP TreeExplainer
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test.values)
print("SHAP values computed.")

# ── Plot 1: Beeswarm (summary plot) ──
plt.figure()
shap.summary_plot(shap_values, X_test.values,
                  feature_names=feature_names,
                  show=False, max_display=20)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIG, "shap_beeswarm.png"), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: shap_beeswarm.png")

# ── Plot 2: Mean absolute SHAP (bar) ──
plt.figure()
shap.summary_plot(shap_values, X_test.values,
                  feature_names=feature_names,
                  plot_type='bar', show=False, max_display=20)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIG, "shap_bar.png"), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: shap_bar.png")

# ── Top 10 features by mean |SHAP| ──
mean_shap = np.abs(shap_values).mean(axis=0)
shap_df   = pd.DataFrame({'Feature': feature_names, 'Mean_SHAP': mean_shap})
shap_df   = shap_df.sort_values('Mean_SHAP', ascending=False).reset_index(drop=True)
print("\nTop 10 Features by Mean |SHAP|:")
print(shap_df.head(10).to_string(index=False))
shap_df.to_csv(os.path.join(OUTPUT_FIG, "shap_feature_importance.csv"), index=False)
print("\nSaved: shap_feature_importance.csv")
print("\nSHAP DONE")

In [ ]:
# ============================================================
# CELL: Racial Fairness Analysis
# ============================================================
import numpy as np
import pandas as pd
import pickle, os, warnings
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score
warnings.filterwarnings('ignore')

PROCESSED  = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")
OUTPUT_FIG = os.path.join(os.getcwd(), "figures")

X_train = pd.read_csv(os.path.join(PROCESSED, "X_train.csv"))
X_test  = pd.read_csv(os.path.join(PROCESSED, "X_test.csv"))
y_train = pd.read_csv(os.path.join(PROCESSED, "y_train.csv")).values.ravel()
y_test  = pd.read_csv(os.path.join(PROCESSED, "y_test.csv")).values.ravel()
demo    = pd.read_csv(os.path.join(PROCESSED, "test_demographics.csv"))

# Retrain XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=1,
    eval_metric='logloss',
    random_state=42, n_jobs=-1
)
xgb_model.fit(X_train.values, y_train)
probs = xgb_model.predict_proba(X_test.values)[:, 1]
preds = (probs >= 0.5).astype(int)

# Attach predictions to demographics
demo = demo.reset_index(drop=True)
demo['prob']  = probs
demo['pred']  = preds
demo['label'] = y_test

# Identify race columns (one-hot encoded)
race_cols = [c for c in demo.columns if c.startswith('race_')]

if race_cols:
    # One-hot → single race label
    demo['race_group'] = demo[race_cols].idxmax(axis=1).str.replace('race_', '')
else:
    demo['race_group'] = demo.get('race', 'Unknown')

print("Race groups found:", demo['race_group'].value_counts().to_dict())

# Per-group metrics (only groups with ≥30 samples)
rows = []
for group, grp in demo.groupby('race_group'):
    if len(grp) < 30:
        continue
    if grp['label'].nunique() < 2:
        continue
    auc  = round(roc_auc_score(grp['label'], grp['prob']), 4)
    acc  = round(accuracy_score(grp['label'], grp['pred']), 4)
    sens = round(recall_score(grp['label'], grp['pred'], zero_division=0), 4)
    spec = round(recall_score(grp['label'], grp['pred'], pos_label=0, zero_division=0), 4)
    rows.append({'Race': group, 'N': len(grp),
                 'AUC-ROC': auc, 'Accuracy': acc,
                 'Sensitivity': sens, 'Specificity': spec})

fairness_df = pd.DataFrame(rows).sort_values('AUC-ROC', ascending=False)
print("\nFairness Analysis — AUC-ROC by Race:")
print(fairness_df.to_string(index=False))
fairness_df.to_csv(os.path.join(OUTPUT_FIG, "fairness_analysis.csv"), index=False)
print("\nSaved: fairness_analysis.csv")

# ── Bar chart ──
plt.figure(figsize=(8, 4))
bars = plt.bar(fairness_df['Race'], fairness_df['AUC-ROC'], color='steelblue', edgecolor='white')
plt.axhline(y=fairness_df['AUC-ROC'].mean(), color='red',
            linestyle='--', label=f"Mean AUC = {fairness_df['AUC-ROC'].mean():.4f}")
plt.ylim(0.5, 1.0)
plt.xlabel('Race Group')
plt.ylabel('AUC-ROC')
plt.title('Model Performance by Race (XGBoost)')
plt.xticks(rotation=30, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIG, "fairness_auc_by_race.png"), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: fairness_auc_by_race.png")
print("\nFAIRNESS DONE")

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os, shap, torch
from sklearn.metrics import roc_curve, auc, roc_auc_score, confusion_matrix

SAVE_DIR = os.path.join(os.getcwd(), "figures")
os.makedirs(SAVE_DIR, exist_ok=True)

def get_proba(model, X):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X.values if hasattr(X, 'values') else np.array(X), dtype=torch.float32)
        out = model(X_t).squeeze()
        if out.dim() == 0:
            out = out.unsqueeze(0)
        probs = torch.sigmoid(out)
    return probs.cpu().numpy()

# Figure 4.1 - SHAP Bar Chart
print("Generating Figure 4.1...")
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)
plt.figure(figsize=(9, 5))
shap.summary_plot(shap_values, X_test, plot_type="bar", max_display=10, show=False)
plt.title("Figure 4.1: Top 10 Features by Mean |SHAP| Value\n(XGBoost - TreeSHAP on Test Set)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "fig4_1_shap_bar.png"), dpi=150, bbox_inches='tight')
plt.close()
print("  fig4_1_shap_bar.png saved")

# Figure 4.2 - SHAP Beeswarm
print("Generating Figure 4.2...")
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values, X_test, plot_type="dot", max_display=10, show=False)
plt.title("Figure 4.2: SHAP Beeswarm Plot - XGBoost (Top 10 Features)\nRed = high feature value  |  Blue = low feature value", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "fig4_2_shap_beeswarm.png"), dpi=150, bbox_inches='tight')
plt.close()
print("  fig4_2_shap_beeswarm.png saved")

# Figure 4.3 - ROC Curves
print("Generating Figure 4.3...")
model_dict = {
    'XGBoost': xgb_model,
    'MLP': mlp_final,
    'SVM': svm_final,
    'LSTM': best_lstm,
    'TCN': best_tcn,
    'TabFCN': tabfcn_final,
    'TabNet': tabnet_final,
    'LR': lr_final,
}
colors_roc = ['#C0392B', '#2980B9', '#27AE60', '#8E44AD', '#E67E22', '#16A085', '#2C3E50', '#95A5A6']
fig, ax = plt.subplots(figsize=(8, 7))
for (name, model), color in zip(model_dict.items(), colors_roc):
    try:
        y_prob = get_proba(model, X_test)
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        ax.plot(fpr, tpr, color=color, lw=1.8, label=name + ' (AUC = ' + str(round(auc(fpr, tpr), 4)) + ')')
    except Exception as e:
        print("  Skipping " + name + ": " + str(e))
ax.plot([0, 1], [0, 1], 'k--', lw=1.0, alpha=0.5, label='Random chance (AUC = 0.50)')
ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=11)
ax.set_title('Figure 4.3: ROC Curves - All 8 Models on Hold-out Test Set', fontsize=12, fontweight='bold')
ax.legend(fontsize=8.5, loc='lower right', framealpha=0.9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "fig4_3_roc_curves.png"), dpi=150, bbox_inches='tight')
plt.close()
print("  fig4_3_roc_curves.png saved")

# Figure 4.4 - Racial Fairness
print("Generating Figure 4.4...")
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_pred_xgb = xgb_model.predict(X_test)
races = ['Hispanic', 'White', 'Other', 'Asian', 'Black']
race_cols = ['Race_Hispanic', 'Race_White', 'Race_Other', 'Race_Asian', 'Race_Black']
auc_vals = []
sens_vals = []
spec_vals = []
labels = []
for race, col in zip(races, race_cols):
    if col not in X_test.columns:
        continue
    mask = X_test[col] == 1
    if mask.sum() < 20:
        continue
    yt = y_test[mask] if hasattr(y_test, 'iloc') else np.array(y_test)[mask]
    if len(np.unique(yt)) < 2:
        continue
    yp = y_prob_xgb[mask]
    yc = y_pred_xgb[mask]
    tn, fp, fn, tp = confusion_matrix(yt, yc).ravel()
    auc_vals.append(roc_auc_score(yt, yp))
    sens_vals.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
    spec_vals.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
    labels.append(race + '\n(n=' + str(int(mask.sum())) + ')')
x = np.arange(len(labels))
width = 0.26
fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - width, auc_vals, width, label='AUC-ROC', color='#2980B9', edgecolor='white')
b2 = ax.bar(x, sens_vals, width, label='Sensitivity', color='#E05C5C', edgecolor='white')
b3 = ax.bar(x + width, spec_vals, width, label='Specificity', color='#27AE60', edgecolor='white')
overall_auc = roc_auc_score(y_test, y_prob_xgb)
ax.axhline(overall_auc, color='#2980B9', lw=1.2, linestyle='--', alpha=0.7, label='Overall AUC-ROC (' + str(round(overall_auc, 4)) + ')')
for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                str(round(bar.get_height(), 3)), ha='center', va='bottom', fontsize=7.5, rotation=90)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Score', fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_title('Figure 4.4: XGBoost Model Performance by Racial Subgroup\n(AUC-ROC, Sensitivity, Specificity)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "fig4_4_fairness.png"), dpi=150, bbox_inches='tight')
plt.close()
print("  fig4_4_fairness.png saved")

print("All 4 figures saved to: " + SAVE_DIR)

In [ ]:
%whos

In [ ]:
# Find your LSTM and TCN model variables
for name in ['lstm_model','lstm_final','lstm_clf','best_lstm','study_lstm',
             'tcn_model','tcn_final','tcn_clf','best_tcn','study_tcn',
             'tabfcn_model','tabfcn_final','best_tabfcn','study_tabfcn',
             'lr_model','lr_final','best_lr','logistic_model']:
    try:
        v = eval(name)
        print(f"✅ {name}  —  {type(v).__name__}")
    except:
        print(f"❌ {name}")

In [ ]:
for name in ['tabnet_model', 'tabnet_final', 'best_tabnet', 'study_tabnet', 'TabNet_model']:
    try:
        v = eval(name)
        print("FOUND: " + name + " -- " + type(v).__name__)
    except:
        print("missing: " + name)

In [ ]:
print([c for c in X_test.columns if 'race' in c.lower()])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_auc_score, confusion_matrix
import os

SAVE_DIR = os.path.join(os.getcwd(), "figures")

y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_pred_xgb = xgb_model.predict(X_test)

races     = ['Black',        'Hispanic',        'Other',        'White']
race_cols = ['race_black', 'race_hispanic', 'race_other', 'race_white']

auc_vals = []
sens_vals = []
spec_vals = []
labels = []

for race, col in zip(races, race_cols):
    if col not in X_test.columns:
        continue
    mask = X_test[col] == 1
    if mask.sum() < 20:
        continue
    yt = y_test[mask] if hasattr(y_test, 'iloc') else np.array(y_test)[mask]
    if len(np.unique(yt)) < 2:
        continue
    yp = y_prob_xgb[mask]
    yc = y_pred_xgb[mask]
    tn, fp, fn, tp = confusion_matrix(yt, yc).ravel()
    auc_vals.append(roc_auc_score(yt, yp))
    sens_vals.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
    spec_vals.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
    labels.append(race + '\n(n=' + str(int(mask.sum())) + ')')

x = np.arange(len(labels))
width = 0.26
fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - width, auc_vals, width, label='AUC-ROC', color='#2980B9', edgecolor='white')
b2 = ax.bar(x, sens_vals, width, label='Sensitivity', color='#E05C5C', edgecolor='white')
b3 = ax.bar(x + width, spec_vals, width, label='Specificity', color='#27AE60', edgecolor='white')
overall_auc = roc_auc_score(y_test, y_prob_xgb)
ax.axhline(overall_auc, color='#2980B9', lw=1.2, linestyle='--', alpha=0.7, label='Overall AUC-ROC (' + str(round(overall_auc, 4)) + ')')
for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                str(round(bar.get_height(), 3)), ha='center', va='bottom', fontsize=7.5, rotation=90)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Score', fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_title('Figure 4.4: XGBoost Model Performance by Racial Subgroup\n(AUC-ROC, Sensitivity, Specificity)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "fig4_4_fairness.png"), dpi=150, bbox_inches='tight')
plt.close()
print("fig4_4_fairness.png saved")

In [ ]:
# ── Paste this cell at the bottom of your training notebook and run it ────────
# It loads the results your notebook already saved as .pkl files and
# combines them into one CSV at:
#   processed/model_results.csv

import pickle
import pandas as pd
from pathlib import Path

OUTPUT_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")

pkl_files = [
    'results_lr.pkl',
    'results_xgb.pkl',
    'results_mlp.pkl',
    'results_tabfcn.pkl',
    'results_svm.pkl',
    'results_tabnet.pkl',
    'results_tcn.pkl',
    'results_lstm.pkl',
]

rows = []
for fname in pkl_files:
    p = Path(OUTPUT_PATH) / fname
    if p.exists():
        with open(p, 'rb') as f:
            rows.append(pickle.load(f))
        print(f'✓ Loaded {fname}')
    else:
        print(f'✗ Not found: {fname}')

df = pd.DataFrame(rows)
out = Path(OUTPUT_PATH) / 'model_results.csv'
df.to_csv(out, index=False)
print(f'\n✓ Saved to {out}')
print(df[['Model','AUC_ROC','Accuracy','F1','Sensitivity','Specificity']].to_string(index=False))

In [ ]:
# Paste into your notebook and run — finds TCN/LSTM pkl files and saves complete CSV

import pickle
import pandas as pd
from pathlib import Path
import os

OUTPUT_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")

# Find ALL pkl files in your processed folder
print("All pkl files found:")
for f in Path(OUTPUT_PATH).glob("*.pkl"):
    print(f"  {f.name}")

print()

# Load the 6 we already know
known = ['results_lr.pkl','results_xgb.pkl','results_mlp.pkl',
         'results_tabfcn.pkl','results_svm.pkl','results_tabnet.pkl']

rows = []
for fname in known:
    p = Path(OUTPUT_PATH) / fname
    if p.exists():
        with open(p,'rb') as f:
            rows.append(pickle.load(f))

# Try every remaining pkl file — if it has 'AUC_ROC' key it's a results file
for f in Path(OUTPUT_PATH).glob("*.pkl"):
    if f.name not in known:
        try:
            with open(f,'rb') as fh:
                obj = pickle.load(fh)
            if isinstance(obj, dict) and 'AUC_ROC' in obj:
                print(f"Found results in: {f.name}  Model={obj.get('Model','?')}")
                rows.append(obj)
        except:
            pass

df = pd.DataFrame(rows).sort_values('AUC_ROC', ascending=False).reset_index(drop=True)
out = Path(OUTPUT_PATH) / 'model_results.csv'
df.to_csv(out, index=False)
print(f'\n✓ Saved {len(df)} models to {out}')
print(df[['Model','AUC_ROC','Accuracy','F1','Sensitivity','Specificity']].to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np

# ── Cohort size ──────────────────────────────────────────────────────────────
print("=" * 50)
print("COHORT STATISTICS")
print("=" * 50)

# Check what dataframes exist - try common variable names
for name in ['df', 'data', 'cohort', 'df_final', 'df_clean', 'X', 'dataset']:
    try:
        obj = eval(name)
        print(f"\n'{name}' shape: {obj.shape}")
        if 'label' in obj.columns:
            print(obj['label'].value_counts())
        elif 'fibroid' in obj.columns:
            print(obj['fibroid'].value_counts())
        elif 'target' in obj.columns:
            print(obj['target'].value_counts())
        elif 'outcome' in obj.columns:
            print(obj['outcome'].value_counts())
    except:
        pass

# ── Train/val/test split sizes ───────────────────────────────────────────────
print("\n" + "=" * 50)
print("SPLIT SIZES")
print("=" * 50)

for name in ['X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test']:
    try:
        obj = eval(name)
        print(f"{name}: {len(obj)}")
    except:
        pass

# ── Class counts in each split ───────────────────────────────────────────────
print("\n" + "=" * 50)
print("CLASS COUNTS PER SPLIT")
print("=" * 50)

for name in ['y_train', 'y_val', 'y_test']:
    try:
        obj = eval(name)
        vals = pd.Series(obj).value_counts().sort_index()
        print(f"\n{name}:")
        print(f"  Controls (0): {vals.get(0, 'N/A')}")
        print(f"  Cases    (1): {vals.get(1, 'N/A')}")
    except:
        pass

In [ ]:
# Get original cohort size before SMOTE
for name in ['df_model', 'df_filtered', 'df_cohort', 'X_raw', 'X_original', 
             'y_raw', 'y_original', 'y', 'labels']:
    try:
        obj = eval(name)
        print(f"'{name}' shape/len: {getattr(obj, 'shape', len(obj))}")
        if hasattr(obj, 'value_counts'):
            print(pd.Series(obj).value_counts().sort_index())
    except:
        pass

# Also try
try:
    total = len(X_test) + len(X_val) + (len(X_train) // 2)  # undo SMOTE approx
    print(f"\nEstimated original N (approx): {total}")
except:
    pass

try:
    print(f"\nX before SMOTE shape: {X_before_smote.shape}")
except:
    pass

In [ ]:
# Get the actual pre-SMOTE counts precisely
print("Val set total:", len(X_val))
print("Test set total:", len(X_test))

# Val cases + test cases
val_cases = (pd.Series(y_val) == 1).sum()
test_cases = (pd.Series(y_test) == 1).sum()
val_controls = (pd.Series(y_val) == 0).sum()
test_controls = (pd.Series(y_test) == 0).sum()

print(f"\nVal:  {val_controls} controls + {val_cases} cases = {len(y_val)}")
print(f"Test: {test_controls} controls + {test_cases} cases = {len(y_test)}")

# Since SMOTE balanced to 50/50 in training, original training cases = half of y_train
original_train_cases = len(y_train) // 2
original_train_controls = len(y_train) // 2  # roughly, before SMOTE

# But actual original training cases = same ratio as val/test
case_rate = (val_cases + test_cases) / (len(y_val) + len(y_test))
original_train_total_approx = int(len(X_train) * case_rate / (case_rate) + len(X_train) * (1 - case_rate))

# Simplest: original train cases should equal val cases * (70/15)
original_train_cases_est = round(val_cases * (70/15))
original_train_controls_est = round(val_controls * (70/15))
original_train_total = original_train_cases_est + original_train_controls_est

print(f"\nEstimated original training set: {original_train_total}")
print(f"Estimated TOTAL original cohort: {original_train_total + len(y_val) + len(y_test)}")
print(f"Estimated total CASES: {original_train_cases_est + val_cases + test_cases}")
print(f"Estimated total CONTROLS: {original_train_controls_est + val_controls + test_controls}")

In [ ]:
# Get the actual ICD code values used
import pandas as pd

for name in ['diagnoses', 'df_diag', 'diag']:
    try:
        obj = eval(name)
        # Find columns that might contain ICD codes
        for col in obj.columns:
            mask = obj[col].astype(str).str.startswith('D25')
            if mask.sum() > 0:
                print(f"Column '{col}' in '{name}':")
                print(obj[col][mask].unique())
        break
    except:
        pass

In [ ]:
import pandas as pd

data_path = os.path.join(os.getcwd(), 'data')

adm = pd.read_csv(f'{data_path}\\admissions.csv', usecols=['admittime'])
print("Min:", pd.to_datetime(adm['admittime']).min())
print("Max:", pd.to_datetime(adm['admittime']).max())

In [ ]:
import os
import glob

# Search everywhere for pkl files
results = glob.glob(os.path.join(os.getcwd(), 'processed', '**', '*.pkl'), recursive=True)
for r in results:
    print(r)

In [ ]:
import pickle

path = os.path.join(os.getcwd(), 'processed', 'results_xgb.pkl')

with open(path, 'rb') as f:
    xgb = pickle.load(f)

for key, val in xgb.items():
    print(f"{key}: {str(val)[:200]}")
    

In [ ]:
import pickle
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, confusion_matrix

# Load xgb results
path = os.path.join(os.getcwd(), 'processed', 'results_xgb.pkl')
with open(path, 'rb') as f:
    xgb = pickle.load(f)

print("All keys:", list(xgb.keys()))

# Check if y_test and race info is stored
for key in xgb.keys():
    print(f"{key}: {str(xgb[key])[:100]}")

In [ ]:
# Load your test set with race info
# Try these variable names
for name in ['X_test', 'df_test', 'test_df', 'X_test_raw']:
    try:
        obj = eval(name)
        print(f"'{name}' columns:", obj.columns.tolist()[:10])
        break
    except:
        pass

# Check if race is in X_test
if 'race' in X_test.columns or any('race' in c.lower() for c in X_test.columns):
    race_col = [c for c in X_test.columns if 'race' in c.lower()][0]
    print("Race column:", race_col)
    print(X_test[race_col].value_counts())

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve

# Use the model already loaded in memory from your notebook
# Try these variable names one by one to find your model
for name in ['xgb_model', 'model', 'best_model', 'clf']:
    try:
        m = eval(name)
        print(f"Found model as '{name}': {type(m)}")
    except:
        pass

# Also check what threshold y_pred used directly
# by comparing against test probabilities at 0.5
y_pred_proba_test = xgb_results.get('y_pred_proba', None)
print("\ny_pred_proba in pkl:", y_pred_proba_test)

# Check if y_pred matches 0.5 threshold using stored probabilities
# If you have y_score or similar
for key in xgb_results.keys():
    val = xgb_results[key]
    print(f"{key}: {type(val)} = {val if not hasattr(val, '__len__') else f'length {len(val)}'}")

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve

# Get validation set probabilities
y_pred_proba_val = xgb_model.predict_proba(X_val)[:, 1]

# Calculate Youden threshold
fpr, tpr, thresholds = roc_curve(y_val, y_pred_proba_val)
youden = tpr - fpr
best_threshold = thresholds[np.argmax(youden)]
print(f"Best Youden threshold on validation set: {best_threshold:.4f}")

# Check if stored y_pred matches 0.5 or Youden threshold
y_pred_proba_test = xgb_model.predict_proba(X_test)[:, 1]
y_pred_at_05 = (y_pred_proba_test >= 0.5).astype(int)
y_pred_at_youden = (y_pred_proba_test >= best_threshold).astype(int)

match_05 = np.mean(y_pred == y_pred_at_05)
match_youden = np.mean(y_pred == y_pred_at_youden)
print(f"y_pred matches 0.5 threshold: {match_05:.4f}")
print(f"y_pred matches Youden threshold: {match_youden:.4f}")

In [ ]:
import numpy as np

y_pred_proba_test = xgb_model.predict_proba(X_test)[:, 1]

# Test a range of thresholds
for t in [0.1, 0.15, 0.1624, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]:
    y_at_t = (y_pred_proba_test >= t).astype(int)
    match = np.mean(y_pred == y_at_t)
    print(f"Threshold {t:.4f}: match = {match:.4f}")

# Also try xgb default predict (uses 0.5 internally)
y_default = xgb_model.predict(X_test)
match_default = np.mean(y_pred == y_default)
print(f"\nxgb_model.predict() default: match = {match_default:.4f}")

In [ ]:
# This will tell us exactly how y_pred was created
import inspect

# Check if y_pred exists and how it was likely created
print("y_pred dtype:", y_pred.dtype)
print("y_pred unique values:", np.unique(y_pred))
print("y_pred value counts - 0:", (y_pred==0).sum(), "| 1:", (y_pred==1).sum())

# Check sensitivity and specificity manually at different thresholds
from sklearn.metrics import confusion_matrix
y_pred_proba_test = xgb_model.predict_proba(X_test)[:, 1]

for t in np.arange(0.05, 0.55, 0.01):
    y_at_t = (y_pred_proba_test >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_at_t).ravel()
    sens = tp/(tp+fn)
    spec = tn/(tn+fp)
    # We know stored sensitivity=0.4967, specificity=0.8436
    if abs(sens - 0.4967) < 0.001 and abs(spec - 0.8436) < 0.001:
        print(f"\nEXACT MATCH at threshold: {t:.4f}")
        print(f"Sensitivity: {sens:.4f}, Specificity: {spec:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

y_pred_proba_test = xgb_model.predict_proba(X_test)[:, 1]

print("Searching for matching threshold...")
for t in np.arange(0.05, 0.55, 0.01):
    y_at_t = (y_pred_proba_test >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_at_t).ravel()
    sens = tp/(tp+fn)
    spec = tn/(tn+fp)
    print(f"t={t:.2f}: sens={sens:.4f}, spec={spec:.4f}")

In [ ]:
import os
print("Notebook is running from:", os.getcwd())

In [ ]:
# ============================================================
# DESCRIPTIVE STATISTICS + CSV EXPORT
# Run this in your Jupyter notebook (fibroid_ai_final env)
# Place this in a new cell at the END of your existing notebook
# ============================================================

import pandas as pd
import numpy as np
import pickle
from scipy import stats
import os

# ----------------------------------------------------------
# 1. LOAD ALL SAVED FILES
# ----------------------------------------------------------
# Adjust these paths to wherever your files are saved
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")

# Load scaler
with open(os.path.join(DATA_DIR, "scaler.pkl"), "rb") as f:
    scaler = pickle.load(f)

# Load feature column names
with open(os.path.join(DATA_DIR, "feature_cols.pkl"), "rb") as f:
    feature_cols = pickle.load(f)

# Load scaled data splits
X_train_scaled = pd.read_csv(os.path.join(DATA_DIR, "X_train.csv"))
X_val_scaled   = pd.read_csv(os.path.join(DATA_DIR, "X_val.csv"))
X_test_scaled  = pd.read_csv(os.path.join(DATA_DIR, "X_test.csv"))

y_train = pd.read_csv(os.path.join(DATA_DIR, "y_train.csv"))
y_val   = pd.read_csv(os.path.join(DATA_DIR, "y_val.csv"))
y_test  = pd.read_csv(os.path.join(DATA_DIR, "y_test.csv"))

print(f"X_train shape: {X_train_scaled.shape}  (NOTE: includes SMOTE synthetic rows)")
print(f"X_val   shape: {X_val_scaled.shape}")
print(f"X_test  shape: {X_test_scaled.shape}")
print(f"y_train value counts:\n{y_train.iloc[:,0].value_counts()}")

# ----------------------------------------------------------
# 2. INVERSE TRANSFORM — RECOVER ORIGINAL CLINICAL VALUES
# ----------------------------------------------------------
X_train_orig = pd.DataFrame(
    scaler.inverse_transform(X_train_scaled.values),
    columns=feature_cols
)
X_val_orig = pd.DataFrame(
    scaler.inverse_transform(X_val_scaled.values),
    columns=feature_cols
)
X_test_orig = pd.DataFrame(
    scaler.inverse_transform(X_test_scaled.values),
    columns=feature_cols
)

# Attach labels
X_train_orig["label"] = y_train.iloc[:, 0].values
X_val_orig["label"]   = y_val.iloc[:, 0].values
X_test_orig["label"]  = y_test.iloc[:, 0].values

print(f"\nAfter inverse transform:")
print(f"  Age range in X_train: {X_train_orig['age'].min():.1f} – {X_train_orig['age'].max():.1f}")
print(f"  Age range in X_test:  {X_test_orig['age'].min():.1f} – {X_test_orig['age'].max():.1f}")

# ----------------------------------------------------------
# 3. ISOLATE REAL (PRE-SMOTE) TRAINING ROWS
#    SMOTE only adds synthetic CASE rows.
#    Real training cases = 2,991 * 0.70 = ~2,094 rows
#    Real training controls = 53,476 * 0.70 = ~37,433 rows
#    Total real train rows should be ~39,527
#
#    Strategy: Keep all controls (not affected by SMOTE) +
#    keep only the first N_real_cases case rows
#    (SMOTE appends synthetic rows at the END)
# ----------------------------------------------------------
# Known cohort totals
TOTAL_N       = 56467
TOTAL_CASES   = 2991
TOTAL_CONTROLS = 53476

# Expected real split sizes (70/15/15)
n_train_real_cases    = round(TOTAL_CASES * 0.70)      # ~2094
n_train_controls      = round(TOTAL_CONTROLS * 0.70)   # ~37433
n_train_real_total    = n_train_real_cases + n_train_controls

print(f"\nExpected real training rows:")
print(f"  Cases (70% of {TOTAL_CASES}): {n_train_real_cases}")
print(f"  Controls (70% of {TOTAL_CONTROLS}): {n_train_controls}")
print(f"  Total: {n_train_real_total}")

# Split X_train_orig into cases and controls
train_cases_df    = X_train_orig[X_train_orig["label"] == 1].reset_index(drop=True)
train_controls_df = X_train_orig[X_train_orig["label"] == 0].reset_index(drop=True)

print(f"\nActual X_train cases: {len(train_cases_df)}  (SMOTE-augmented)")
print(f"Actual X_train controls: {len(train_controls_df)}  (real only)")

# Keep only the first n_train_real_cases case rows (real patients before SMOTE appended)
train_real_cases_df    = train_cases_df.iloc[:n_train_real_cases]
train_real_df          = pd.concat([train_real_cases_df, train_controls_df], ignore_index=True)

print(f"\nReal training rows after removing SMOTE: {len(train_real_df)}")
print(f"  Cases: {(train_real_df['label']==1).sum()}")
print(f"  Controls: {(train_real_df['label']==0).sum()}")

# ----------------------------------------------------------
# 4. FULL COHORT = real train + val + test
# ----------------------------------------------------------
full_cohort = pd.concat([train_real_df, X_val_orig, X_test_orig], ignore_index=True)
print(f"\nFull cohort size: {len(full_cohort)}")
print(f"  Cases: {(full_cohort['label']==1).sum()}")
print(f"  Controls: {(full_cohort['label']==0).sum()}")

# ----------------------------------------------------------
# 5. SAVE ALL DATASETS AS CSV (no PKL)
# ----------------------------------------------------------
train_real_df.to_csv(os.path.join(DATA_DIR, "X_train_original_values.csv"), index=False)
X_val_orig.to_csv(   os.path.join(DATA_DIR, "X_val_original_values.csv"),   index=False)
X_test_orig.to_csv(  os.path.join(DATA_DIR, "X_test_original_values.csv"),  index=False)
full_cohort.to_csv(  os.path.join(DATA_DIR, "full_cohort_original_values.csv"), index=False)

print("\n✓ Saved:")
print("  X_train_original_values.csv  — real pre-SMOTE training patients")
print("  X_val_original_values.csv    — validation patients")
print("  X_test_original_values.csv   — test patients")
print("  full_cohort_original_values.csv — all 56,467 patients")

# ----------------------------------------------------------
# 6. DESCRIPTIVE STATISTICS TABLE
# ----------------------------------------------------------
df = full_cohort.copy()
cases_mask    = df["label"] == 1
controls_mask = df["label"] == 0

def fmt_mean_sd(series):
    return f"{series.mean():.1f} ± {series.std():.1f}"

def fmt_median_iqr(series):
    return f"{series.median():.1f} [{series.quantile(0.25):.1f}–{series.quantile(0.75):.1f}]"

def fmt_n_pct(n, total):
    return f"{int(n)} ({n/total*100:.1f}%)"

total_n       = len(df)
total_cases   = cases_mask.sum()
total_controls = controls_mask.sum()

print("\n" + "="*70)
print("TABLE: BASELINE CHARACTERISTICS OF THE STUDY COHORT")
print("="*70)
print(f"{'Characteristic':<30} {'Cases (N={:,})'.format(total_cases):<22} {'Controls (N={:,})'.format(total_controls):<22} {'Total (N={:,})'.format(total_n)}")
print("-"*70)

# Continuous
for col, name, unit in [
    ("age",         "Age",         "years"),
    ("bmi",         "BMI",         "kg/m²"),
    ("haemoglobin", "Haemoglobin", "g/dL"),
    ("inr",         "INR",         ""),
]:
    c  = df.loc[cases_mask, col]
    ct = df.loc[controls_mask, col]
    a  = df[col]
    t_stat, p_val = stats.ttest_ind(c, ct)
    p_str = f"p<0.001" if p_val < 0.001 else f"p={p_val:.3f}"
    print(f"{name} ({unit}), mean±SD")
    print(f"  {'':28} {fmt_mean_sd(c):<22} {fmt_mean_sd(ct):<22} {fmt_mean_sd(a)}  ({p_str})")
    print(f"  Median [IQR]{'':20} {fmt_median_iqr(c):<22} {fmt_median_iqr(ct):<22} {fmt_median_iqr(a)}")

print()
print("Race/Ethnicity, n (%)")
for col, race in [
    ("race_black",    "  Black"),
    ("race_white",    "  White"),
    ("race_other",    "  Other"),
    ("race_hispanic", "  Hispanic"),
]:
    n_c  = int(round(df.loc[cases_mask, col].sum()))
    n_ct = int(round(df.loc[controls_mask, col].sum()))
    n_a  = int(round(df[col].sum()))
    print(f"{race:<30} {fmt_n_pct(n_c, total_cases):<22} {fmt_n_pct(n_ct, total_controls):<22} {fmt_n_pct(n_a, total_n)}")

# Asian = no other race flag
asian = ((df["race_black"].round()==0) & (df["race_hispanic"].round()==0) &
         (df["race_other"].round()==0) & (df["race_white"].round()==0))
n_c  = asian[cases_mask].sum()
n_ct = asian[controls_mask].sum()
n_a  = asian.sum()
print(f"{'  Asian':<30} {fmt_n_pct(n_c, total_cases):<22} {fmt_n_pct(n_ct, total_controls):<22} {fmt_n_pct(n_a, total_n)}")

print()
print("Comorbidities, n (%)")
for col, name in [("hypertension","  Hypertension"), ("diabetes","  Diabetes")]:
    n_c  = int(round(df.loc[cases_mask, col].round().sum()))
    n_ct = int(round(df.loc[controls_mask, col].round().sum()))
    n_a  = int(round(df[col].round().sum()))
    chi2, p_val = stats.chi2_contingency(
        [[n_c, total_cases-n_c],[n_ct, total_controls-n_ct]]
    )[:2]
    p_str = f"p<0.001" if p_val < 0.001 else f"p={p_val:.3f}"
    print(f"{name:<30} {fmt_n_pct(n_c, total_cases):<22} {fmt_n_pct(n_ct, total_controls):<22} {fmt_n_pct(n_a, total_n)}  ({p_str})")

print("="*70)
print("\nNote: 'Other' includes unspecified/unknown race categories in MIMIC-IV.")
print("Statistical tests: independent-samples t-test for continuous variables,")
print("chi-square test for categorical variables.")

# ----------------------------------------------------------
# 7. ALSO SAVE DESCRIPTIVE STATS AS CSV
# ----------------------------------------------------------
rows = []

for col, name, unit in [
    ("age",         "Age (years)",       "years"),
    ("bmi",         "BMI (kg/m²)",       "kg/m²"),
    ("haemoglobin", "Haemoglobin (g/dL)","g/dL"),
    ("inr",         "INR",               ""),
]:
    c  = df.loc[cases_mask, col]
    ct = df.loc[controls_mask, col]
    a  = df[col]
    t_stat, p_val = stats.ttest_ind(c, ct)
    rows.append({
        "Variable": name,
        "Cases_Mean": round(c.mean(), 2),
        "Cases_SD": round(c.std(), 2),
        "Cases_Median": round(c.median(), 2),
        "Cases_IQR_25": round(c.quantile(0.25), 2),
        "Cases_IQR_75": round(c.quantile(0.75), 2),
        "Controls_Mean": round(ct.mean(), 2),
        "Controls_SD": round(ct.std(), 2),
        "Controls_Median": round(ct.median(), 2),
        "Controls_IQR_25": round(ct.quantile(0.25), 2),
        "Controls_IQR_75": round(ct.quantile(0.75), 2),
        "Overall_Mean": round(a.mean(), 2),
        "Overall_SD": round(a.std(), 2),
        "p_value": round(p_val, 4),
        "Type": "continuous"
    })

for col, race in [
    ("race_black",    "Race: Black"),
    ("race_white",    "Race: White"),
    ("race_other",    "Race: Other"),
    ("race_hispanic", "Race: Hispanic"),
]:
    n_c  = int(round(df.loc[cases_mask, col].sum()))
    n_ct = int(round(df.loc[controls_mask, col].sum()))
    n_a  = int(round(df[col].sum()))
    rows.append({
        "Variable": race,
        "Cases_Mean": n_c, "Cases_SD": round(n_c/total_cases*100,1),
        "Controls_Mean": n_ct, "Controls_SD": round(n_ct/total_controls*100,1),
        "Overall_Mean": n_a, "Overall_SD": round(n_a/total_n*100,1),
        "Type": "categorical_n_pct"
    })

for col, name in [("hypertension","Hypertension"), ("diabetes","Diabetes")]:
    n_c  = int(round(df.loc[cases_mask, col].round().sum()))
    n_ct = int(round(df.loc[controls_mask, col].round().sum()))
    n_a  = int(round(df[col].round().sum()))
    chi2, p_val = stats.chi2_contingency(
        [[n_c, total_cases-n_c],[n_ct, total_controls-n_ct]]
    )[:2]
    rows.append({
        "Variable": name,
        "Cases_Mean": n_c, "Cases_SD": round(n_c/total_cases*100,1),
        "Controls_Mean": n_ct, "Controls_SD": round(n_ct/total_controls*100,1),
        "Overall_Mean": n_a, "Overall_SD": round(n_a/total_n*100,1),
        "p_value": round(p_val, 4),
        "Type": "categorical_n_pct"
    })

desc_df = pd.DataFrame(rows)
desc_df.to_csv(os.path.join(DATA_DIR, "descriptive_statistics.csv"), index=False)
print("\n✓ Saved: descriptive_statistics.csv")
print("\nAll done!")

In [ ]:
import pickle, os
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")
with open(os.path.join(DATA_DIR, "feature_cols.pkl"), "rb") as f:
    feature_cols = pickle.load(f)
print(feature_cols)

In [ ]:
import pandas as pd, pickle, os
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")
X_train = pd.read_csv(os.path.join(DATA_DIR, "X_train.csv"))
print(X_train.isnull().mean().sort_values(ascending=False) * 100)

In [ ]:
import os
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")
for f in sorted(os.listdir(DATA_DIR)):
    print(f)

In [ ]:
import pandas as pd, os
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")

df = pd.read_csv(os.path.join(DATA_DIR, "all_results_summary.csv"))
print(df.to_string(index=False))

In [ ]:
import pandas as pd, os
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")

df = pd.read_csv(os.path.join(DATA_DIR, "model_results.csv"))
print(df.to_string(index=False))

In [ ]:
import pandas as pd, os
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")
demo = pd.read_csv(os.path.join(DATA_DIR, "test_demographics.csv"))
print(demo.groupby('race')['label'].agg(['sum','count']))

In [ ]:
import pickle, os
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")
with open(os.path.join(DATA_DIR, "results_xgb.pkl"), 'rb') as f:
    res = pickle.load(f)
print(res.keys())

In [ ]:
import os
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")
for f in sorted(os.listdir(DATA_DIR)):
    if 'xgb' in f.lower() or 'boost' in f.lower() or '.pt' in f.lower() or '.json' in f.lower() or '.bin' in f.lower() or '.pkl' in f.lower():
        print(f)

In [ ]:
import gc
# Find all model-like variables in memory
for name, obj in list(globals().items()):
    t = type(obj).__name__
    if any(x in t.lower() for x in ['classifier', 'xgb', 'mlp', 'svm', 'lgbm', 'net', 'model', 'tcn', 'lstm', 'tabnet', 'logistic']):
        print(name, '→', t)

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import pandas as pd, os, numpy as np
import torch
import torch.nn.functional as F
import shap

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")
SAVE_DIR = os.path.join(os.getcwd(), "figures")

X_test = pd.read_csv(os.path.join(DATA_DIR, "X_test.csv"))
y_test = pd.read_csv(os.path.join(DATA_DIR, "y_test.csv")).iloc[:, 0].values
print(f"Test set loaded: {X_test.shape[0]} rows, {X_test.shape[1]} features")

# ── probability helper ────────────────────────────────────────────────────────
def get_proba(model, X):
    model_name = type(model).__name__.lower()
    # TabNet needs numpy
    if 'tabnet' in model_name:
        return model.predict_proba(X.values)[:, 1]
    # sklearn / XGBoost / LR
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    # PyTorch (MLP, TCN, LSTM)
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X.values, dtype=torch.float32)
        if 'tcn' in model_name:
            X_t = X_t.unsqueeze(-1)
        elif 'lstm' in model_name:
            X_t = X_t.unsqueeze(1)
        out = model(X_t)
        if out.dim() == 2 and out.shape[1] == 2:
            p = F.softmax(out, dim=1)[:, 1]
        else:
            p = torch.sigmoid(out).squeeze()
        return p.cpu().numpy()

# ── FIGURE 4.1 — ROC curves ───────────────────────────────────────────────────
models_roc = {
    'XGBoost': xgb_final,
    'MLP':     mlp_final,
    'TCN':     best_tcn,
    'LSTM':    best_lstm,
    'TabNet':  tabnet_final,
    'LR':      lr_final,
}
colors = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b']

plt.figure(figsize=(10, 8))
for (name, model), color in zip(models_roc.items(), colors):
    p = get_proba(model, X_test)
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc(fpr,tpr):.4f})')
plt.plot([0,1],[0,1],'k--', lw=1.5, label='Random chance (AUC = 0.5000)')
plt.xlim([0,1]); plt.ylim([0,1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=13)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=13)
plt.title('ROC Curves — All Models on Hold-out Test Set (N = 8,471)', fontsize=14)
plt.legend(loc='lower right', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig4_3_roc_curves.png'), dpi=300, bbox_inches='tight')
plt.show(); print("✓ Figure 4.1 ROC saved")

# ── FIGURES 4.2 & 4.3 — SHAP bar + beeswarm ─────────────────────────────────
explainer   = shap.Explainer(xgb_final, X_test)
shap_values = explainer(X_test)

plt.figure(figsize=(10, 7))
shap.plots.bar(shap_values, max_display=24, show=False)
plt.title('Global SHAP Feature Importance (Mean |SHAP Value|, XGBoost)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig4_1_shap_bar.png'), dpi=300, bbox_inches='tight')
plt.show(); print("✓ Figure 4.2 SHAP bar saved")

plt.figure(figsize=(10, 8))
shap.plots.beeswarm(shap_values, max_display=24, show=False)
plt.title('SHAP Beeswarm Plot — Individual Feature Contributions (XGBoost)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig4_2_shap_beeswarm.png'), dpi=300, bbox_inches='tight')
plt.show(); print("✓ Figure 4.3 SHAP beeswarm saved")

# ── FIGURE 4.4 — fairness grouped bar ────────────────────────────────────────
groups = ['Hispanic', 'Other', 'Asian', 'White', 'Black']
sens   = [0.500,      0.344,   0.242,   0.141,   0.464]
spec   = [0.903,      0.883,   0.912,   0.960,   0.784]

x = np.arange(len(groups)); w = 0.35
fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - w/2, sens, w, label='Sensitivity', color='#2196F3')
bars2 = ax.bar(x + w/2, spec, w, label='Specificity', color='#FF9800')
ax.set_xlabel('Racial / Ethnic Group', fontsize=13)
ax.set_ylabel('Metric Value', fontsize=13)
ax.set_title('Sensitivity and Specificity by Racial Group\n(XGBoost, Test Partition, N = 8,471)', fontsize=13)
ax.set_xticks(x); ax.set_xticklabels(groups, fontsize=12)
ax.set_ylim(0, 1.05); ax.legend(fontsize=12)
ax.axhline(np.mean(sens), color='#2196F3', linestyle='--', alpha=0.5)
ax.axhline(np.mean(spec), color='#FF9800', linestyle='--', alpha=0.5)
for bar in bars1: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
for bar in bars2: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig4_4_fairness.png'), dpi=300, bbox_inches='tight')
plt.show(); print("✓ Figure 4.4 fairness saved")

print("\n✅ All 4 figures saved to", SAVE_DIR)

In [ ]:
import os
print([f for f in os.listdir(os.path.join(os.getcwd(), "data")) if 'questionnaire' in f.lower()])

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

FILE = os.path.join(os.getcwd(), "data", "Questionnaire.csv")

df = pd.read_csv(FILE)

cols = ['Timestamp','Aged18+','UKResident','Consent','Age','Ethnicity','Marital','Height','Weight',
        'FibroidDx','WhenDx','HowDx','SymptomsBefore','Symptoms','HeavyBleeding','BleedingDuration',
        'Anaemia','Comorbidities','VisitedHCP','VisitCount','SuggestedFibroids','SymptomDuration',
        'EverPregnant','EverBirth','FreeText']
df.columns = cols
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

cases    = df[df['FibroidDx'] == 'Yes']
controls = df[df['FibroidDx'] == 'No']

print("=" * 50)
print(f"TOTAL RESPONSES: {len(df)}")
print(f"Cases (fibroid Yes): {len(cases)}")
print(f"Controls (fibroid No): {len(controls)}")

print("\n--- AGE ---")
print(f"Overall mean age: {df['Age'].mean():.1f}, range {df['Age'].min():.0f}–{df['Age'].max():.0f}")
print(f"Cases mean age: {cases['Age'].mean():.1f} (SD {cases['Age'].std():.1f})")
print(f"Controls mean age: {controls['Age'].mean():.1f} (SD {controls['Age'].std():.1f})")

print("\n--- ETHNICITY ---")
print("Overall:"); print(df['Ethnicity'].value_counts())
print("\nCases:"); print(cases['Ethnicity'].value_counts())
print("\nControls:"); print(controls['Ethnicity'].value_counts())

print("\n--- ANAEMIA ---")
print(f"Cases with anaemia: {len(cases[cases['Anaemia']=='Yes'])}/12 = {len(cases[cases['Anaemia']=='Yes'])/12*100:.0f}%")
print(f"Controls with anaemia: {len(controls[controls['Anaemia']=='Yes'])}/12 = {len(controls[controls['Anaemia']=='Yes'])/12*100:.0f}%")

print("\n--- HEAVY BLEEDING (cases) ---")
print(f"Yes: {len(cases[cases['HeavyBleeding']=='Yes'])}/12 = {len(cases[cases['HeavyBleeding']=='Yes'])/12*100:.0f}%")

print("\n--- DIAGNOSTIC DELAY (cases) ---")
print(cases['SymptomDuration'].value_counts())
over1yr = cases[cases['SymptomDuration'].isin(['1–2 years','More than 2 years'])]
print(f"Delay > 1 year: {len(over1yr)}/12 = {len(over1yr)/12*100:.0f}%")
over2yr = cases[cases['SymptomDuration'] == 'More than 2 years']
print(f"Delay > 2 years: {len(over2yr)}/12 = {len(over2yr)/12*100:.0f}%")

print("\n--- HCP DID NOT SUGGEST FIBROIDS (cases) ---")
no_suggest = len(cases[cases['SuggestedFibroids']=='No'])
print(f"{no_suggest}/12 = {no_suggest/12*100:.0f}%")

print("\n--- VISITED HCP BEFORE DIAGNOSIS (cases) ---")
visited = len(cases[cases['VisitedHCP']=='Yes'])
print(f"{visited}/12 = {visited/12*100:.0f}%")

print("\n" + "=" * 50)
print("DONE")

In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.metrics import roc_auc_score
from scipy import stats

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "processed")
X_test = pd.read_csv(os.path.join(DATA_DIR, "X_test.csv"))
y_test = pd.read_csv(os.path.join(DATA_DIR, "y_test.csv")).iloc[:, 0].values

# ── get probabilities for all models ─────────────────────────────────────────
def get_proba(model, X):
    model_name = type(model).__name__.lower()
    if 'tabnet' in model_name:
        return model.predict_proba(X.values)[:, 1]
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    import torch, torch.nn.functional as F
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X.values, dtype=torch.float32)
        if 'tcn' in model_name:
            X_t = X_t.unsqueeze(-1)
        elif 'lstm' in model_name:
            X_t = X_t.unsqueeze(1)
        out = model(X_t)
        if out.dim() == 2 and out.shape[1] == 2:
            p = torch.nn.functional.softmax(out, dim=1)[:, 1]
        else:
            p = torch.sigmoid(out).squeeze()
        return p.cpu().numpy()

models_dict = {
    'XGBoost': xgb_final,
    'MLP':     mlp_final,
    'TCN':     best_tcn,
    'LSTM':    best_lstm,
    'TabNet':  tabnet_final,
    'LR':      lr_final,
    'SVM':     svm_final,      # add this
    'TabFCN':  tabfcn_final,   # add this
}

# ── bootstrapped 95% CI for AUC-ROC ─────────────────────────────────────────
def bootstrap_auc_ci(y_true, y_prob, n_bootstrap=1000, ci=0.95):
    rng = np.random.RandomState(42)
    aucs = []
    n = len(y_true)
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y_true[idx], y_prob[idx]))
    lower = np.percentile(aucs, (1 - ci) / 2 * 100)
    upper = np.percentile(aucs, (1 + ci) / 2 * 100)
    return np.mean(aucs), lower, upper

# ── DeLong test between two models ───────────────────────────────────────────
def delong_test(y_true, prob_a, prob_b):
    # Hanley & McNeil (1983) approximation
    n1 = np.sum(y_true == 1)
    n0 = np.sum(y_true == 0)
    auc_a = roc_auc_score(y_true, prob_a)
    auc_b = roc_auc_score(y_true, prob_b)
    
    def auc_variance(y_true, prob):
        pos = prob[y_true == 1]
        neg = prob[y_true == 0]
        auc = roc_auc_score(y_true, prob)
        q1 = auc / (2 - auc)
        q2 = 2 * auc**2 / (1 + auc)
        var = (auc*(1-auc) + (n1-1)*(q1-auc**2) + (n0-1)*(q2-auc**2)) / (n1*n0)
        return var
    
    var_a = auc_variance(y_true, prob_a)
    var_b = auc_variance(y_true, prob_b)
    
    # correlation between the two AUCs (simplified)
    r = 0.5  # conservative estimate
    se_diff = np.sqrt(var_a + var_b - 2*r*np.sqrt(var_a*var_b))
    z = (auc_a - auc_b) / se_diff
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return auc_a, auc_b, z, p

# ── run everything ────────────────────────────────────────────────────────────
print("=" * 65)
print("BOOTSTRAPPED 95% CONFIDENCE INTERVALS FOR AUC-ROC (N=1000)")
print("=" * 65)

probas = {}
for name, model in models_dict.items():
    p = get_proba(model, X_test)
    probas[name] = p
    _, lo, hi = bootstrap_auc_ci(y_test, p)
    auc = roc_auc_score(y_test, p)
    print(f"{name:10s}  AUC = {auc:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")
print("\n" + "=" * 65)
print("DeLong PAIRWISE TESTS vs XGBoost (best model)")
print("=" * 65)
ref_name = 'XGBoost'
for name in models_dict:
    if name == ref_name:
        continue
    auc_a, auc_b, z, p = delong_test(y_test, probas[ref_name], probas[name])
    sig = "significant" if p < 0.05 else "not significant"
    print(f"XGBoost vs {name:10s}  z={z:.3f}  p={p:.4f}  → {sig}")

print("\nAll results saved to processed/ folder.")
